In [1]:
#| default_exp core

# Core asterism search module
> Coordinate transforms, scoring functions, grid utilities, and GPU pipeline

In [2]:
#| export
import os

# CRITICAL: Set LD_LIBRARY_PATH BEFORE importing torch (for HIP kernel)
torch_lib_path = os.path.expanduser("~") + "/dev/amateur_astro/py-asterisms/.venv/lib/python3.12/site-packages/torch/lib"
if "LD_LIBRARY_PATH" in os.environ:
    if torch_lib_path not in os.environ["LD_LIBRARY_PATH"]:
        os.environ["LD_LIBRARY_PATH"] = f"{torch_lib_path}:{os.environ['LD_LIBRARY_PATH']}"
else:
    os.environ["LD_LIBRARY_PATH"] = torch_lib_path

# Fix GPU memory fragmentation for ROCm/HIP
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import polars as pl
import numpy as np
import torch
import math
import time
import heapq
from typing import List, Any
import tqdm
import pickle
import pyarrow as pa
from dataclasses import dataclass, field
import hashlib

## Search configuration

| Mode | Max Mag | Max Extent | Grid Step | Search Radius |
|------|---------|------------|-----------|---------------|
| Naked eye | 6 | 20deg | 10deg | 20deg |
| Binocular | 11 | 6deg | 4deg | 6deg |
| Telescopic | 15 | 2deg | 4deg | 2deg |

In [ ]:
#| export
@dataclass
class SearchConfig:
    name: str
    max_mag: float
    max_extent_deg: float
    grid_step_deg: float
    search_radius_deg: float
    max_stars_per_region: int = 0  # 0 = no cap

NAKED_EYE = SearchConfig("naked_eye", 6, 20, 10, 20)
BINOCULAR = SearchConfig("binocular", 11, 6, 4, 6, max_stars_per_region=300)
TELESCOPIC = SearchConfig("telescopic", 15, 2, 4, 2)

@dataclass
class Eyepiece:
    focal_length_mm: float
    afov_deg: float

@dataclass
class Camera:
    name: str
    pixel_size_um: float
    res_x: int
    res_y: int
    limiting_mag: float

@dataclass
class InstrumentConfig:
    name: str
    aperture_mm: float
    focal_ratio: float
    sqm: float
    eyepieces: list = field(default_factory=list)
    camera: Camera = None

    @property
    def focal_length_mm(self):
        return self.aperture_mm * self.focal_ratio

    def to_dict(self):
        d = {
            "name": self.name,
            "aperture_mm": self.aperture_mm,
            "focal_ratio": self.focal_ratio,
            "sqm": self.sqm,
            "eyepieces": [{"focal_length_mm": ep.focal_length_mm, "afov_deg": ep.afov_deg}
                          for ep in self.eyepieces],
        }
        if self.camera:
            d["camera"] = {
                "name": self.camera.name,
                "pixel_size_um": self.camera.pixel_size_um,
                "res_x": self.camera.res_x,
                "res_y": self.camera.res_y,
                "limiting_mag": self.camera.limiting_mag,
            }
        return d

    @classmethod
    def from_dict(cls, d):
        eyepieces = [Eyepiece(**ep) for ep in d.get("eyepieces", [])]
        camera = Camera(**d["camera"]) if "camera" in d else None
        return cls(
            name=d["name"], aperture_mm=d["aperture_mm"],
            focal_ratio=d["focal_ratio"], sqm=d["sqm"],
            eyepieces=eyepieces, camera=camera,
        )

DEFAULT_INSTRUMENT = InstrumentConfig(
    name="10in_f5",
    aperture_mm=254,
    focal_ratio=5,
    sqm=20,
    eyepieces=[
        Eyepiece(5, 65),
        Eyepiece(7, 86),
        Eyepiece(13, 100),
        Eyepiece(20, 86),
        Eyepiece(38, 70),
    ],
)

def camera_fov(cam, focal_length_mm):
    """Compute camera sensor FOV in degrees. Returns (fov_w, fov_h)."""
    sensor_w = cam.res_x * cam.pixel_size_um / 1000
    sensor_h = cam.res_y * cam.pixel_size_um / 1000
    fov_w = 2 * math.degrees(math.atan(sensor_w / (2 * focal_length_mm)))
    fov_h = 2 * math.degrees(math.atan(sensor_h / (2 * focal_length_mm)))
    return fov_w, fov_h

def sqm_to_nelm(sqm):
    """SQM reading to naked-eye limiting magnitude."""
    return 7.93 - 5 * math.log10(10**(4.316 - sqm / 5) + 1)

def telescope_limiting_mag(aperture_mm, exit_pupil_mm, nelm):
    """Visual limiting magnitude for telescope + eyepiece combo."""
    ep = max(exit_pupil_mm, 0.5)
    lm = nelm + 5 * math.log10(aperture_mm / ep)
    return min(lm, 2.7 + 5 * math.log10(aperture_mm))

def eyepiece_to_search_config(inst, ep, max_stars_per_region=200):
    """Derive a SearchConfig from instrument + eyepiece physical specs."""
    magnification = inst.focal_length_mm / ep.focal_length_mm
    tfov = ep.afov_deg / magnification
    exit_pupil = min(ep.focal_length_mm / inst.focal_ratio, 7.0)
    nelm = sqm_to_nelm(inst.sqm)
    lm = telescope_limiting_mag(inst.aperture_mm, exit_pupil, nelm)
    search_radius = round(tfov * 0.85, 2)
    return SearchConfig(
        name=f"{inst.name}_{ep.focal_length_mm:.0f}mm_mag{round(lm, 1)}",
        max_mag=round(lm, 1),
        max_extent_deg=search_radius,
        grid_step_deg=max(1, round(search_radius * 0.6, 1)),
        search_radius_deg=search_radius,
        max_stars_per_region=max_stars_per_region,
    )

def camera_to_search_config(inst, max_stars_per_region=200):
    """Derive a SearchConfig from instrument + camera sensor specs."""
    fov_w, fov_h = camera_fov(inst.camera, inst.focal_length_mm)
    tfov_short = min(fov_w, fov_h)
    search_radius = round(tfov_short * 0.85, 2)
    return SearchConfig(
        name=f"{inst.name}_{inst.camera.name}_mag{inst.camera.limiting_mag}",
        max_mag=inst.camera.limiting_mag,
        max_extent_deg=search_radius,
        grid_step_deg=max(0.3, round(search_radius * 0.6, 1)),
        search_radius_deg=search_radius,
        max_stars_per_region=max_stars_per_region,
    )

def instrument_search_configs(inst, max_stars_per_region=200):
    """Generate SearchConfigs for all eyepieces and camera in an instrument."""
    configs = [eyepiece_to_search_config(inst, ep, max_stars_per_region) for ep in inst.eyepieces]
    if inst.camera:
        configs.append(camera_to_search_config(inst, max_stars_per_region))
    return configs

## Data classes and coordinate transforms

In [ ]:
#| export
@dataclass(order=True)
class ScoreItem:
    score: float
    region: int=field(compare=False)
    item: Any=field(compare=False)
    
@dataclass(order=True)
class Points:
    points: Any=field(compare=False)
    

def convert_to_cartesian(distances, ra, dec):
    x = distances * torch.cos(dec) * torch.cos(ra)
    y = distances * torch.cos(dec) * torch.sin(ra)
    z = distances * torch.sin(dec)
    return torch.stack((x, y, z), dim=1)

def calculate_distances(coords):
    dist_matrix = torch.cdist(coords, coords)
    unique_distances = torch.unique(dist_matrix)
    return unique_distances[unique_distances > 0]

def distance_from_magnitude(m, M):
    return 10**((m - M + 5) / 5)

def distance_from_magnitude_tensor(m: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    return 10**((m - M + 5) / 5)

def tetrahedron_score(coords):
    distances = calculate_distances(coords)
    print(distances)
    mean_distance = torch.mean(distances)
    std_dev = torch.std(distances)
    score = math.exp(-std_dev.item() / mean_distance.item())
    return score

def square_score(tensor):
    """Return a measure of how close the points in a tensor are to forming a square."""
    spatial_distances = torch.pdist(tensor, p=2)
    print(spatial_distances)
    sorted_distances = torch.sort(spatial_distances)[0]
    std_of_smallest_4 = sorted_distances[:4].std().item()
    return std_of_smallest_4

def measure_squareness_old(tensor):
    spatial_distances = torch.pdist(tensor, p=2)
    distances = torch.sort(spatial_distances)[0]
    mean_sides = torch.mean(distances[:4]).clamp(min=1e-12)
    mean_diagonal = torch.mean(distances[4:])
    squareness = mean_diagonal / mean_sides
    return abs(1 - (squareness / torch.sqrt(torch.tensor(2.0)).item()))

def measure_squareness(tensor):
    spatial_distances = torch.pdist(tensor, p=2)
    distances = torch.sort(spatial_distances)[0]
    std_sides = torch.std(distances[:4])
    std_diagonal = torch.std(distances[4:])
    side_uniformity = torch.exp(-std_sides)
    diagonal_uniformity = torch.exp(-std_diagonal)
    mean_sides = torch.mean(distances[:4])
    mean_diagonal = torch.mean(distances[4:])
    squareness_ratio = mean_sides / mean_diagonal
    final_squareness = (squareness_ratio / torch.sqrt(torch.tensor(2.0)).item()) * side_uniformity * diagonal_uniformity
    return abs(1 - final_squareness.item())

def radecmag_to_cartesian(points_tensor, search_radius_deg=None):
    """Convert (RA_deg, Dec_deg, Vmag) to 3D using perceptual distance from magnitude.
    
    Uses distance modulus with M=0: d = 10^((m+5)/5).
    Normalized so the brightest star (lowest mag) in the group sits at r=1;
    only relative magnitude differences affect the 3D shape.
    
    If search_radius_deg is given, the depth axis is scaled so that the
    magnitude range maps to a depth comparable to the angular extent.
    Without this, small-FOV patterns become needle-like (edge-on) and
    large-FOV patterns become pancake-flat (face-on)."""
    ra_rad = torch.deg2rad(points_tensor[:, 0])
    dec_rad = torch.deg2rad(points_tensor[:, 1])
    mag = points_tensor[:, 2]
    min_mag = mag.min()
    r = 10.0 ** ((mag - min_mag) / 5.0)
    if search_radius_deg is not None and search_radius_deg > 0:
        angular_scale = 2.0 * math.sin(math.radians(search_radius_deg) / 2.0)
        depth_range = r.max() - 1.0
        if depth_range > 1e-6:
            r = 1.0 + (r - 1.0) * (angular_scale / depth_range)
    x = r * torch.cos(dec_rad) * torch.cos(ra_rad)
    y = r * torch.cos(dec_rad) * torch.sin(ra_rad)
    z = r * torch.sin(dec_rad)
    return torch.stack([x, y, z], dim=1)

def radecmag_to_angular(points_tensor):
    """Convert (RA_deg, Dec_deg, Vmag) to unit vectors on celestial sphere.
    
    Chord distance between unit vectors is monotonic with angular separation,
    so equilateral in chord distance = equilateral on the sky."""
    ra_rad = torch.deg2rad(points_tensor[:, 0])
    dec_rad = torch.deg2rad(points_tensor[:, 1])
    x = torch.cos(dec_rad) * torch.cos(ra_rad)
    y = torch.cos(dec_rad) * torch.sin(ra_rad)
    z = torch.sin(dec_rad)
    return torch.stack([x, y, z], dim=1)

## Scoring functions

In [ ]:
#| export
def _pruned_triangle_indices_gpu(scoring_coords, cv_threshold=0.10):
    """GPU-vectorized distance-bucket triangle pruning.
    
    For each pair (i,j) at distance d, find k where dist(i,k)~d AND dist(j,k)~d.
    Reduces C(N,3) to a fraction of candidates while capturing 100% of
    equilateral triangles with cv < threshold.
    
    Falls back to torch.combinations for N <= 300 where brute force is faster.
    """
    N = scoring_coords.shape[0]
    device = scoring_coords.device
    
    if N <= 300:
        return torch.combinations(torch.arange(N, device=device), r=3)
    
    dist = torch.cdist(scoring_coords, scoring_coords)
    
    candidates = []
    for i in range(N - 2):
        dists_i = dist[i]
        j_start = i + 1
        d_ij = dists_i[j_start:N]
        tol = cv_threshold * d_ij
        
        for j_local in range(len(d_ij) - 1):
            j = j_start + j_local
            d = d_ij[j_local]
            if d < 1e-12:
                continue
            t = tol[j_local]
            
            k_slice = slice(j + 1, N)
            mask = (torch.abs(dists_i[k_slice] - d) < t) & (torch.abs(dist[j, k_slice] - d) < t)
            k_hits = torch.where(mask)[0] + (j + 1)
            
            if len(k_hits) > 0:
                triplets = torch.stack([
                    torch.full_like(k_hits, i),
                    torch.full_like(k_hits, j),
                    k_hits
                ], dim=1)
                candidates.append(triplets)
    
    if not candidates:
        return torch.zeros((0, 3), dtype=torch.long, device=device)
    return torch.cat(candidates, dim=0)


def _pruned_square_indices(scoring_coords, tol_frac=0.20):
    """GPU rotation-based square candidate finder.
    
    For each pair (i,j) treated as a diagonal, rotate the half-diagonal by 90°
    to predict where the other two vertices should be. Find nearest stars to
    those predicted positions. O(N^2) pairs × O(N) nearest-neighbor on GPU.
    
    Captures ~90-95% of top squares with 5000-50000x candidate reduction.
    Falls back to torch.combinations for N <= 80.
    """
    N = scoring_coords.shape[0]
    device = scoring_coords.device
    
    if N <= 80:
        return torch.combinations(torch.arange(N, device=device), r=4)
    
    coords_2d = scoring_coords[:, :2] if scoring_coords.shape[1] > 2 else scoring_coords
    
    ii, jj = torch.triu_indices(N, N, offset=1, device=device)
    n_pairs = ii.shape[0]
    
    mid = (coords_2d[ii] + coords_2d[jj]) / 2
    half_diag = (coords_2d[jj] - coords_2d[ii]) / 2
    hd_len = torch.linalg.norm(half_diag, dim=1)
    
    # Rotate half-diagonal by 90° to predict other two vertices
    rotated = torch.stack([-half_diag[:, 1], half_diag[:, 0]], dim=1)
    expected_k = mid + rotated
    expected_l = mid - rotated
    
    batch_size = 50000
    all_quads_i = []
    all_quads_j = []
    all_quads_k = []
    all_quads_l = []
    
    for start in range(0, n_pairs, batch_size):
        end = min(start + batch_size, n_pairs)
        
        dist_k = torch.cdist(expected_k[start:end], coords_2d)
        dist_l = torch.cdist(expected_l[start:end], coords_2d)
        
        tol = tol_frac * hd_len[start:end]
        
        min_dk, nearest_k = dist_k.min(dim=1)
        min_dl, nearest_l = dist_l.min(dim=1)
        
        mask = (min_dk < tol) & (min_dl < tol)
        
        if mask.any():
            valid = torch.where(mask)[0]
            p = valid + start
            qi = ii[p]
            qj = jj[p]
            qk = nearest_k[valid]
            ql = nearest_l[valid]
            
            # Filter: 4 distinct vertices
            distinct = (qi != qk) & (qi != ql) & (qj != qk) & (qj != ql) & (qk != ql)
            if distinct.any():
                d_idx = torch.where(distinct)[0]
                all_quads_i.append(qi[d_idx])
                all_quads_j.append(qj[d_idx])
                all_quads_k.append(qk[d_idx])
                all_quads_l.append(ql[d_idx])
    
    if not all_quads_i:
        return torch.zeros((0, 4), dtype=torch.long, device=device)
    
    qi = torch.cat(all_quads_i)
    qj = torch.cat(all_quads_j)
    qk = torch.cat(all_quads_k)
    ql = torch.cat(all_quads_l)
    
    # Sort vertices per quad and deduplicate
    quads = torch.stack([qi, qj, qk, ql], dim=1)
    quads = quads.sort(dim=1).values
    quads = torch.unique(quads, dim=0)
    
    return quads


def mass_score_triangle_torch(points_tensor, device='cpu', mode='3d', search_radius_deg=None):
    points_tensor = points_tensor.to(device)

    if mode == '3d':
        scoring_coords = radecmag_to_cartesian(points_tensor, search_radius_deg=search_radius_deg)
    else:
        scoring_coords = radecmag_to_angular(points_tensor)

    idx_combinations = _pruned_triangle_indices_gpu(scoring_coords)

    if len(idx_combinations) == 0:
        empty_scores = torch.zeros(0, device=device)
        empty_points = torch.zeros((0, 3, 3), device=device)
        return empty_scores, empty_points

    p1 = scoring_coords[idx_combinations[:, 0]]
    p2 = scoring_coords[idx_combinations[:, 1]]
    p3 = scoring_coords[idx_combinations[:, 2]]

    a = torch.linalg.norm(p2 - p1, dim=1)
    b = torch.linalg.norm(p3 - p2, dim=1)
    c = torch.linalg.norm(p1 - p3, dim=1)

    # Raw coefficient of variation (scale-invariant)
    mean = (a + b + c) / 3
    std_dev = torch.sqrt(((a - mean)**2 + (b - mean)**2 + (c - mean)**2) / 3)
    scores = torch.where(mean > 0, std_dev / mean, torch.tensor([1.], device=mean.device))

    # Return original RA/Dec/Mag points for storage
    p1_full = points_tensor[idx_combinations[:, 0]]
    p2_full = points_tensor[idx_combinations[:, 1]]
    p3_full = points_tensor[idx_combinations[:, 2]]
    points_combined = torch.stack([p1_full, p2_full, p3_full], dim=1)
    return scores, points_combined


# The 3 distinct quadrilateral cycles for vertices {0,1,2,3}.
# Each cycle defines 4 side-pairs and 2 diagonal-pairs.
_QUAD_CYCLES = [
    # cycle 0-1-2-3: sides (01,12,23,30), diags (02,13)
    ([0,1], [1,2], [2,3], [3,0], [0,2], [1,3]),
    # cycle 0-1-3-2: sides (01,13,32,20), diags (03,12)
    ([0,1], [1,3], [3,2], [2,0], [0,3], [1,2]),
    # cycle 0-2-1-3: sides (02,21,13,30), diags (01,23)
    ([0,2], [2,1], [1,3], [3,0], [0,1], [2,3]),
]


def mass_score_square_torch(points_tensor, device='cpu', mode='3d', search_radius_deg=None):
    """Score square candidates, vectorized on GPU.
    
    Uses rotation-based diagonal pruning for N>80 to reduce O(N^4) to O(N^2).
    Tests all 3 quadrilateral vertex orderings per candidate."""
    points_tensor = points_tensor.to(device)

    if mode == '3d':
        scoring_coords = radecmag_to_cartesian(points_tensor, search_radius_deg=search_radius_deg)
    else:
        scoring_coords = radecmag_to_angular(points_tensor)

    idx_combinations = _pruned_square_indices(scoring_coords)

    if len(idx_combinations) == 0:
        empty_scores = torch.zeros(0, device=device)
        empty_points = torch.zeros((0, 4, 3), device=device)
        return empty_scores, empty_points

    # Get all 4-point groups: (N_combos, 4, D)
    combos = scoring_coords[idx_combinations]
    N = combos.shape[0]
    sqrt2 = math.sqrt(2.0)

    best_scores = torch.full((N,), float('inf'), device=device)

    for cycle in _QUAD_CYCLES:
        s0, s1, s2, s3, d0, d1 = cycle
        side_dists = torch.stack([
            torch.linalg.norm(combos[:, s0[0]] - combos[:, s0[1]], dim=1),
            torch.linalg.norm(combos[:, s1[0]] - combos[:, s1[1]], dim=1),
            torch.linalg.norm(combos[:, s2[0]] - combos[:, s2[1]], dim=1),
            torch.linalg.norm(combos[:, s3[0]] - combos[:, s3[1]], dim=1),
        ], dim=1)  # (N, 4)
        diag_dists = torch.stack([
            torch.linalg.norm(combos[:, d0[0]] - combos[:, d0[1]], dim=1),
            torch.linalg.norm(combos[:, d1[0]] - combos[:, d1[1]], dim=1),
        ], dim=1)  # (N, 2)

        mean_sides = side_dists.mean(dim=1)
        mean_diags = diag_dists.mean(dim=1)
        cv_sides = side_dists.std(dim=1) / mean_sides.clamp(min=1e-12)
        cv_diags = diag_dists.std(dim=1) / mean_diags.clamp(min=1e-12)
        ratio_dev = torch.abs(mean_diags / mean_sides.clamp(min=1e-12) - sqrt2)
        cycle_scores = cv_sides + cv_diags + ratio_dev

        best_scores = torch.minimum(best_scores, cycle_scores)

    # Return original RA/Dec/Mag points for storage
    points_combined = points_tensor[idx_combinations]  # (N_combos, 4, 3)
    return best_scores, points_combined


def transform_radecmag_from_numpy(stars):
    torch_tensors = [torch.from_numpy(star) for star in stars]
    zeroes = torch.zeros(len(torch_tensors[2]))
    print("one ", torch_tensors)
    torch_tensors[2] = distance_from_magnitude_tensor(torch_tensors[2], zeroes)
    print("two", torch_tensors)
    coords = convert_to_cartesian(*torch_tensors)
    return coords

def global_normalize_tensor(tensor):
    """Normalize a tensor based on its global min and max values."""
    global_min = torch.min(tensor)
    global_max = torch.max(tensor)
    normalized = (tensor - global_min) / (global_max - global_min)
    return normalized
    
def radec_normalize_tensor(tensors):
    """Normalize tensors based on their global min and max values, excluding the 3rd column."""
    tensor = tensors[:, :2]
    global_min = torch.min(tensor)
    global_max = torch.max(tensor)
    normalized = (tensor - global_min) / (global_max - global_min)
    return normalized

def mag_score(tensor):
    max = tensor[:, 2].max()
    min = tensor[:, 2].min()
    return max - min

def score_triangle(tensor):    
    spatial_distances = torch.pdist(tensor, p=2)
    return torch.std(spatial_distances)

## Angular extent filtering

In [ ]:
#| export
def triangle_extent_deg(points_tensor):
    """Max pairwise angular separation in degrees for a triangle's 3 vertices.
    
    Args:
        points_tensor: shape (3, 3) tensor of (RA_deg, Dec_deg, Vmag)
    Returns:
        Maximum angular separation in degrees between any two vertices.
    """
    ra = torch.deg2rad(points_tensor[:, 0])
    dec = torch.deg2rad(points_tensor[:, 1])
    
    # Pairwise angular separation using spherical law of cosines
    max_sep = 0.0
    for i in range(3):
        for j in range(i+1, 3):
            cos_sep = (torch.sin(dec[i]) * torch.sin(dec[j]) +
                       torch.cos(dec[i]) * torch.cos(dec[j]) * torch.cos(ra[i] - ra[j]))
            cos_sep = torch.clamp(cos_sep, -1.0, 1.0)
            sep = torch.acos(cos_sep)
            max_sep = max(max_sep, torch.rad2deg(sep).item())
    return max_sep


def triangle_extent_deg_batch(points_batch):
    """Vectorized max pairwise angular separation for a batch of triangles.
    
    Args:
        points_batch: shape (N, 3, 3) tensor of (RA_deg, Dec_deg, Vmag)
    Returns:
        Tensor of shape (N,) with max angular separation in degrees per triangle.
    """
    ra = torch.deg2rad(points_batch[:, :, 0])   # (N, 3)
    dec = torch.deg2rad(points_batch[:, :, 1])   # (N, 3)
    
    # Pairs: (0,1), (0,2), (1,2)
    pairs = [(0, 1), (0, 2), (1, 2)]
    seps = []
    for i, j in pairs:
        cos_sep = (torch.sin(dec[:, i]) * torch.sin(dec[:, j]) +
                   torch.cos(dec[:, i]) * torch.cos(dec[:, j]) * torch.cos(ra[:, i] - ra[:, j]))
        cos_sep = torch.clamp(cos_sep, -1.0, 1.0)
        seps.append(torch.acos(cos_sep))
    
    all_seps = torch.stack(seps, dim=1)  # (N, 3)
    return torch.rad2deg(all_seps.max(dim=1).values)


def shape_extent_deg_batch(points_batch):
    """Vectorized max pairwise angular separation for a batch of any shape.
    
    Args:
        points_batch: shape (N, K, 3) tensor where K=3 (triangle) or K=4 (square)
    Returns:
        Tensor of shape (N,) with max angular separation in degrees.
    """
    K = points_batch.shape[1]
    ra = torch.deg2rad(points_batch[:, :, 0])   # (N, K)
    dec = torch.deg2rad(points_batch[:, :, 1])   # (N, K)
    
    seps = []
    for i in range(K):
        for j in range(i + 1, K):
            cos_sep = (torch.sin(dec[:, i]) * torch.sin(dec[:, j]) +
                       torch.cos(dec[:, i]) * torch.cos(dec[:, j]) * torch.cos(ra[:, i] - ra[:, j]))
            cos_sep = torch.clamp(cos_sep, -1.0, 1.0)
            seps.append(torch.acos(cos_sep))
    
    all_seps = torch.stack(seps, dim=1)
    return torch.rad2deg(all_seps.max(dim=1).values)



def compute_tilt_batch(points_batch, search_radius_deg=None):
    """Tilt angle of a shape relative to line of sight: 0° = face-on, 90° = edge-on.

    Depth (magnitude axis) is scaled per-shape to match the shape's own angular
    extent so that tilt is meaningful regardless of FOV size.

    Args:
        points_batch: (N, K, 3) tensor of (RA_deg, Dec_deg, Vmag), K=3 or K=4
        search_radius_deg: enables per-shape depth scaling (any truthy value activates it)
    Returns:
        (N,) tensor of tilt angles in degrees.
    """
    N, K, _ = points_batch.shape
    device = points_batch.device

    flat = points_batch.reshape(N * K, 3)
    ra_rad = torch.deg2rad(flat[:, 0])
    dec_rad = torch.deg2rad(flat[:, 1])
    mag = flat[:, 2].reshape(N, K)
    min_mag = mag.min(dim=1, keepdim=True).values
    r = 10.0 ** ((mag - min_mag) / 5.0)
    if search_radius_deg is not None:
        extents_deg = shape_extent_deg_batch(points_batch)
        angular_scale = 2.0 * torch.sin(torch.deg2rad(extents_deg) / 2.0)
        angular_scale = angular_scale.unsqueeze(1)  # (N, 1)
        depth_range = r.max(dim=1, keepdim=True).values - 1.0
        scale = torch.where(depth_range > 1e-6, angular_scale / depth_range,
                            torch.ones_like(depth_range))
        r = 1.0 + (r - 1.0) * scale
    r_flat = r.reshape(N * K)

    x = r_flat * torch.cos(dec_rad) * torch.cos(ra_rad)
    y = r_flat * torch.cos(dec_rad) * torch.sin(ra_rad)
    z = r_flat * torch.sin(dec_rad)
    cart = torch.stack([x, y, z], dim=1).reshape(N, K, 3)

    if K == 3:
        normal = torch.cross(cart[:, 1] - cart[:, 0], cart[:, 2] - cart[:, 0], dim=1)
    elif K == 4:
        normal = torch.cross(cart[:, 2] - cart[:, 0], cart[:, 3] - cart[:, 1], dim=1)
    else:
        return torch.zeros(N, device=device)

    view = cart.mean(dim=1)

    normal_hat = normal / (torch.linalg.norm(normal, dim=1, keepdim=True) + 1e-12)
    view_hat = view / (torch.linalg.norm(view, dim=1, keepdim=True) + 1e-12)

    cos_angle = torch.abs((normal_hat * view_hat).sum(dim=1)).clamp(0.0, 1.0)
    tilt = torch.rad2deg(torch.acos(cos_angle))
    return tilt


def chain_extent_deg(points_tensor):
    """Max pairwise angular separation in degrees for a chain of K stars.
    
    Args:
        points_tensor: shape (K, 3) tensor of (RA_deg, Dec_deg, Vmag), K >= 4
    Returns:
        Maximum angular separation in degrees between any two stars.
    """
    K = len(points_tensor)
    ra = torch.deg2rad(points_tensor[:, 0])
    dec = torch.deg2rad(points_tensor[:, 1])
    max_sep = 0.0
    for i in range(K):
        for j in range(i + 1, K):
            cos_sep = (torch.sin(dec[i]) * torch.sin(dec[j]) +
                       torch.cos(dec[i]) * torch.cos(dec[j]) * torch.cos(ra[i] - ra[j]))
            cos_sep = torch.clamp(cos_sep, -1.0, 1.0)
            sep = torch.acos(cos_sep)
            max_sep = max(max_sep, torch.rad2deg(sep).item())
    return max_sep

## Collinear chain detection

Finds chains of 4+ stars lying on a great circle. Uses angle binning (O(n² log n))
instead of brute-force C(n,4) (O(n⁴)).

In [ ]:
#| export
import gc
from collections import defaultdict


def _get_rss_gb():
    """Current RSS in GB via /proc/self/status."""
    try:
        with open('/proc/self/status') as f:
            for line in f:
                if line.startswith('VmRSS:'):
                    return int(line.split()[1]) / (1024 * 1024)
    except Exception:
        pass
    return 0.0

def _to_unit_vectors(stars_tensor):
    """Convert (RA_deg, Dec_deg, Vmag) to unit vectors on the celestial sphere."""
    ra_rad = torch.deg2rad(stars_tensor[:, 0])
    dec_rad = torch.deg2rad(stars_tensor[:, 1])
    x = torch.cos(dec_rad) * torch.cos(ra_rad)
    y = torch.cos(dec_rad) * torch.sin(ra_rad)
    z = torch.sin(dec_rad)
    return torch.stack([x, y, z], dim=1)


def _to_unit_vectors_batch(stars_batch):
    """Convert (B, N, 3) batch of (RA_deg, Dec_deg, Vmag) to unit vectors."""
    ra_rad = torch.deg2rad(stars_batch[:, :, 0])
    dec_rad = torch.deg2rad(stars_batch[:, :, 1])
    x = torch.cos(dec_rad) * torch.cos(ra_rad)
    y = torch.cos(dec_rad) * torch.sin(ra_rad)
    z = torch.sin(dec_rad)
    return torch.stack([x, y, z], dim=2)


def _perpendicular_distance(unit_vecs, i_start, i_end):
    """RMS perpendicular distance of interior points from the great circle through endpoints.
    
    Returns (rms_perp_deg, max_perp_deg) for all points between start and end.
    """
    normal = torch.cross(unit_vecs[i_start], unit_vecs[i_end], dim=0)
    norm = torch.linalg.norm(normal)
    if norm < 1e-12:
        return 0.0, 0.0
    normal = normal / norm
    interior = [i for i in range(len(unit_vecs)) if i != i_start and i != i_end]
    if len(interior) == 0:
        return 0.0, 0.0
    dots = torch.abs(unit_vecs[interior] @ normal)
    dots = torch.clamp(dots, 0.0, 1.0)
    perp_rad = torch.asin(dots)
    perp_deg = torch.rad2deg(perp_rad)
    return perp_deg.pow(2).mean().sqrt().item(), perp_deg.max().item()


def _batch_compute_angles(padded_stars, counts, device, k_angle=50):
    """Compute direction angles to K nearest neighbors per star.
    
    Uses KNN-bounded approach: O(B*N*K) memory instead of O(B*N²).
    Per-region KNN avoids allocating B*N*N distance matrices.
    
    Args:
        padded_stars: (B, max_N, 3) tensor of stars, zero-padded
        counts: (B,) int tensor of actual star counts per region
        device: GPU device
        k_angle: number of nearest neighbors for angle computation
    Returns:
        sorted_angles: (B, max_N, K) sorted angle values (CPU numpy)
        sorted_indices: (B, max_N, K) sorted star indices (CPU numpy)
        unit_vecs: (B, max_N, 3) unit vectors (stays on GPU)
    """
    B, max_N, _ = padded_stars.shape
    K = min(k_angle, max_N - 1)

    unit_vecs = _to_unit_vectors_batch(padded_stars)  # (B, N, 3)

    # Compute KNN per region to avoid (B, N, N) distance matrix
    all_knn = torch.zeros(B, max_N, K, dtype=torch.long, device=device)
    for b in range(B):
        n = int(counts[b])
        if n < 4:
            continue
        uv_b = unit_vecs[b, :n]
        dots = (uv_b @ uv_b.T).clamp(-1, 1)
        dists_b = torch.sqrt(torch.clamp(2 - 2 * dots, min=0))
        dists_b.fill_diagonal_(float('inf'))
        k_b = min(K, n - 1)
        _, topk_idx = torch.topk(dists_b, k_b, dim=1, largest=False)
        all_knn[b, :n, :k_b] = topk_idx
        del dots, dists_b, topk_idx

    ra_rad = torch.deg2rad(padded_stars[:, :, 0])   # (B, N)
    dec_rad = torch.deg2rad(padded_stars[:, :, 1])   # (B, N)

    # Tangent frames: (B, N, 3)
    east = torch.stack([
        -torch.sin(ra_rad), torch.cos(ra_rad), torch.zeros_like(ra_rad)
    ], dim=2)
    north = torch.stack([
        -torch.sin(dec_rad) * torch.cos(ra_rad),
        -torch.sin(dec_rad) * torch.sin(ra_rad),
        torch.cos(dec_rad),
    ], dim=2)

    # Gather neighbor unit vectors: (B, N, K, 3)
    knn_flat = all_knn.reshape(B, max_N * K)  # (B, N*K)
    neighbor_uv = torch.gather(
        unit_vecs, 1,
        knn_flat.unsqueeze(-1).expand(-1, -1, 3)
    ).reshape(B, max_N, K, 3)
    del knn_flat

    # Direction from star i to each KNN neighbor: (B, N, K, 3)
    diffs = neighbor_uv - unit_vecs.unsqueeze(2)
    del neighbor_uv

    # Project out radial component
    radial_dot = (diffs * unit_vecs.unsqueeze(2)).sum(dim=3, keepdim=True)  # (B, N, K, 1)
    tangent = diffs - radial_dot * unit_vecs.unsqueeze(2)  # (B, N, K, 3)
    del diffs, radial_dot

    # Project onto tangent frames
    tx = (tangent * east.unsqueeze(2)).sum(dim=3)   # (B, N, K)
    ty = (tangent * north.unsqueeze(2)).sum(dim=3)  # (B, N, K)
    del tangent
    angles = torch.atan2(ty, tx) % math.pi          # (B, N, K)
    del tx, ty

    # Mask padding: pi sorts to end
    for b in range(B):
        n = int(counts[b])
        k_b = min(K, n - 1)
        if n < max_N:
            angles[b, n:, :] = math.pi
        if k_b < K:
            angles[b, :n, k_b:] = math.pi

    sorted_angles, sort_perm = torch.sort(angles, dim=2)
    sorted_indices = torch.gather(all_knn, 2, sort_perm)

    return sorted_angles.cpu().numpy(), sorted_indices.cpu().numpy(), unit_vecs


def _find_chains_batch(sa_cpu, si_cpu, counts_cpu, angle_tol_rad):
    """Find collinear chain candidates across all regions.
    
    Uses numpy vectorized diff/cumsum for cluster detection instead of
    Python sliding windows. Only iterates in Python over rows that actually
    contain chains (pre-filtered via vectorized check).
    
    Returns list of (region_b, chain_indices_tuple).
    """
    B, max_N, K = sa_cpu.shape
    all_candidates = []

    for b in range(B):
        N = int(counts_cpu[b])
        if N < 4:
            continue

        M = min(K, N - 1)
        if M < 3:
            continue

        # Extract this region's sorted angles and indices for KNN neighbors
        block_a = sa_cpu[b, :N, :M]  # (N, M)
        block_i = si_cpu[b, :N, :M]  # (N, M)

        # Vectorized cluster detection: diff + cumsum for all rows at once
        diffs = np.diff(block_a, axis=1)  # (N, M-1)
        breaks = diffs > angle_tol_rad    # (N, M-1)

        # Pre-filter: only process rows with runs of 3+ (2+ consecutive non-breaks)
        if M >= 3:
            has_pair = ~breaks[:, :-1] & ~breaks[:, 1:]  # (N, M-2)
            row_has_chain = np.any(has_pair, axis=1)      # (N,)
        else:
            continue

        # Also check wrap-around: angles near 0 and near pi are close
        wrap_gap = block_a[:, 0] + (math.pi - block_a[:, M - 1])
        row_has_wrap = wrap_gap < angle_tol_rad

        active_rows = np.where(row_has_chain | row_has_wrap)[0]
        if len(active_rows) == 0:
            continue

        seen = set()

        for i in active_rows:
            row_a = block_a[i]   # (M,)
            row_idx = block_i[i] # (M,)

            # Cluster IDs via cumsum on breaks
            cluster_ids = np.empty(M, dtype=np.int32)
            cluster_ids[0] = 0
            cluster_ids[1:] = np.cumsum(breaks[i])

            # Handle wrap-around: merge last cluster into first
            if row_has_wrap[i] and cluster_ids[-1] > 0:
                last_cid = cluster_ids[-1]
                cluster_ids[cluster_ids == last_cid] = 0

            # Count cluster sizes via bincount
            max_cid = cluster_ids[-1] + 1
            sizes = np.bincount(cluster_ids, minlength=max_cid)

            # Process clusters with >= 3 members (+ reference star i = chain of 4+)
            large = np.where(sizes >= 3)[0]
            for cid in large:
                members = row_idx[cluster_ids == cid]
                chain = tuple(sorted(np.append(members, i)))
                if chain not in seen:
                    seen.add(chain)
                    all_candidates.append((b, chain))

    return all_candidates



def _find_smooth_chains(uv_np, max_turn_deg=30.0, min_stars=4, k_neighbors=30):
    """Find smooth chains (lines AND curves) using GPU-batched greedy path extension.
    
    Uses KNN-bounded neighbor search for O(N*K) instead of O(N^2) per step.
    
    Args:
        uv_np: (N, 3) numpy array of unit vectors on celestial sphere
        max_turn_deg: maximum turning angle between consecutive segments
        min_stars: minimum chain length
        k_neighbors: number of nearest neighbors for search (default 30)
    
    Returns:
        List of chains, each a tuple of star indices (sorted)
    """
    N = len(uv_np)
    if N < min_stars:
        return []
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    uv = torch.tensor(uv_np, dtype=torch.float32, device=device)
    
    # KNN via topk (avoids full N*N argsort)
    dots = uv @ uv.T
    dists = torch.sqrt(torch.clamp(2 - 2 * dots, min=0))
    del dots
    dists.fill_diagonal_(float('inf'))
    
    k = min(k_neighbors, N - 1)
    _, knn = torch.topk(dists, k, dim=1, largest=False)  # (N, k)
    del dists
    
    cos_max_turn = math.cos(math.radians(max_turn_deg))
    
    # Build all starting pairs: (N*k) chains, each starting with (i, knn[i,j])
    k_start = min(15, k)  # use fewer starting pairs than search neighbors
    knn_start = knn[:, :k_start]
    starts_i = torch.arange(N, device=device).unsqueeze(1).expand(-1, k_start).reshape(-1)
    starts_j = knn_start.reshape(-1)
    
    C = len(starts_i)  # total chains
    max_len = 50  # max chain length
    
    # Chain storage: (C, max_len) indices, -1 = empty
    chains = torch.full((C, max_len), -1, dtype=torch.long, device=device)
    chains[:, 0] = starts_i
    chains[:, 1] = starts_j
    chain_lens = torch.full((C,), 2, dtype=torch.long, device=device)
    
    # Used mask: (C, N) — which stars are used per chain
    used = torch.zeros(C, N, dtype=torch.bool, device=device)
    used[torch.arange(C, device=device), starts_i] = True
    used[torch.arange(C, device=device), starts_j] = True
    
    # Current direction per chain: (C, 3)
    direction = uv[starts_j] - uv[starts_i]
    dir_norm = torch.linalg.norm(direction, dim=1, keepdim=True).clamp(min=1e-12)
    direction = direction / dir_norm
    
    # Current last star per chain
    last = starts_j.clone()
    
    # Active mask
    active = torch.ones(C, dtype=torch.bool, device=device)
    active &= (dir_norm.squeeze() > 1e-12)
    
    # Forward extension — KNN-bounded
    for step in range(max_len - 2):
        if not active.any():
            break
        
        n_active = active.sum().item()
        act_idx = torch.where(active)[0]
        
        last_stars = last[act_idx]                          # (n_active,)
        last_uv = uv[last_stars]                            # (n_active, 3)
        last_knn = knn[last_stars]                          # (n_active, k) global indices
        neighbor_uv = uv[last_knn]                          # (n_active, k, 3)
        
        vecs = neighbor_uv - last_uv.unsqueeze(1)          # (n_active, k, 3)
        norms = torch.linalg.norm(vecs, dim=2)              # (n_active, k)
        norms_safe = norms.clone()
        norms_safe[norms_safe < 1e-12] = float('inf')
        normed = vecs / norms_safe.unsqueeze(2)             # (n_active, k, 3)
        
        act_dir = direction[act_idx]                        # (n_active, 3)
        cos_turns = (normed * act_dir.unsqueeze(1)).sum(dim=2)  # (n_active, k)
        
        neighbor_used = used[act_idx].gather(1, last_knn)   # (n_active, k)
        valid = ~neighbor_used & (cos_turns >= cos_max_turn) & (norms_safe < float('inf'))
        
        big_dist = torch.full_like(norms, float('inf'))
        big_dist[valid] = norms[valid]
        local_nearest = big_dist.argmin(dim=1)              # (n_active,) index into k
        has_valid = valid.any(dim=1)
        
        next_stars_global = last_knn[torch.arange(n_active, device=device), local_nearest]
        
        found = act_idx[has_valid]
        if len(found) == 0:
            break
        
        next_stars = next_stars_global[has_valid]
        cur_lens = chain_lens[found]
        
        chains[found, cur_lens] = next_stars
        chain_lens[found] = cur_lens + 1
        used[found, next_stars] = True
        
        new_dir = normed[has_valid, local_nearest[has_valid]]
        direction[found] = new_dir
        last[found] = next_stars
        
        not_found = act_idx[~has_valid]
        active[not_found] = False
    
    # Backward extension — KNN-bounded
    active_bwd = chain_lens >= 2
    bwd_dir = uv[starts_i] - uv[starts_j]
    bwd_norm = torch.linalg.norm(bwd_dir, dim=1, keepdim=True).clamp(min=1e-12)
    bwd_dir = bwd_dir / bwd_norm
    bwd_last = starts_i.clone()
    
    bwd_chains = torch.full((C, max_len), -1, dtype=torch.long, device=device)
    bwd_lens = torch.zeros(C, dtype=torch.long, device=device)
    
    for step in range(max_len - 2):
        if not active_bwd.any():
            break
        
        act_idx = torch.where(active_bwd)[0]
        n_active = len(act_idx)
        
        last_stars = bwd_last[act_idx]
        last_uv = uv[last_stars]
        last_knn = knn[last_stars]
        neighbor_uv = uv[last_knn]
        
        vecs = neighbor_uv - last_uv.unsqueeze(1)
        norms = torch.linalg.norm(vecs, dim=2)
        norms_safe = norms.clone()
        norms_safe[norms_safe < 1e-12] = float('inf')
        normed = vecs / norms_safe.unsqueeze(2)
        
        act_dir = bwd_dir[act_idx]
        cos_turns = (normed * act_dir.unsqueeze(1)).sum(dim=2)
        
        neighbor_used = used[act_idx].gather(1, last_knn)
        valid = ~neighbor_used & (cos_turns >= cos_max_turn) & (norms_safe < float('inf'))
        
        big_dist = torch.full_like(norms, float('inf'))
        big_dist[valid] = norms[valid]
        local_nearest = big_dist.argmin(dim=1)
        has_valid = valid.any(dim=1)
        
        next_stars_global = last_knn[torch.arange(n_active, device=device), local_nearest]
        
        found = act_idx[has_valid]
        if len(found) == 0:
            break
        
        next_stars = next_stars_global[has_valid]
        cur_bwd_lens = bwd_lens[found]
        
        bwd_chains[found, cur_bwd_lens] = next_stars
        bwd_lens[found] = cur_bwd_lens + 1
        used[found, next_stars] = True
        
        new_dir = normed[has_valid, local_nearest[has_valid]]
        bwd_dir[found] = new_dir
        bwd_last[found] = next_stars
        
        not_found = act_idx[~has_valid]
        active_bwd[not_found] = False
    
    # Assemble full chains and deduplicate
    seen = set()
    result_chains = []
    
    chain_lens_cpu = chain_lens.cpu().numpy()
    bwd_lens_cpu = bwd_lens.cpu().numpy()
    chains_cpu = chains.cpu().numpy()
    bwd_chains_cpu = bwd_chains.cpu().numpy()
    
    for c in range(C):
        fwd_len = chain_lens_cpu[c]
        bwd_len = bwd_lens_cpu[c]
        total_len = fwd_len + bwd_len
        if total_len < min_stars:
            continue
        
        # Assemble: reversed backward + forward
        full = []
        for bi in range(bwd_len - 1, -1, -1):
            full.append(int(bwd_chains_cpu[c, bi]))
        for fi in range(fwd_len):
            full.append(int(chains_cpu[c, fi]))
        
        key = frozenset(full)
        if key not in seen:
            seen.add(key)
            result_chains.append(tuple(full))
    
    return result_chains


SCORE_CUTOFFS = {'rms': 0.2, 'msd': 0.2, 'snake': 1.5}


def _batch_score_chains(chains_uv, chains_mag=None, max_mag=15.0):
    """Batch-score chains of the same length K on GPU.
    
    Args:
        chains_uv: (N, K, 3) tensor of unit vectors
        chains_mag: optional (N, K) tensor of magnitudes
        max_mag: limiting magnitude for normalizing mean_mag (from config)
    Returns:
        scores_dict: dict with 'rms', 'msd', 'smooth', 'smooth_mag' score tensors (N,)
        sort_orders: (N, K) tensor of indices sorting stars along the chain
    """
    N, K, _ = chains_uv.shape
    device = chains_uv.device

    # SVD for principal axis direction
    centroid = chains_uv.mean(dim=1, keepdim=True)
    centered = chains_uv - centroid
    _, _, Vh = torch.linalg.svd(centered, full_matrices=False)
    principal = Vh[:, 0, :]  # (N, 3)

    # Project onto principal axis and sort
    projections = (chains_uv * principal.unsqueeze(1)).sum(dim=2)  # (N, K)
    sort_order = torch.argsort(projections, dim=1)

    batch_idx = torch.arange(N, device=device).unsqueeze(1).expand(-1, K)
    sorted_uv = chains_uv[batch_idx, sort_order]

    # Straightness: RMS perpendicular distance from great circle through endpoints
    normal = torch.cross(sorted_uv[:, 0], sorted_uv[:, -1], dim=1)
    normal_norm = torch.linalg.norm(normal, dim=1, keepdim=True).clamp(min=1e-12)
    normal = normal / normal_norm

    interior = sorted_uv[:, 1:-1]  # (N, K-2, 3)
    dots = torch.abs((interior * normal.unsqueeze(1)).sum(dim=2)).clamp(0, 1)
    perp_deg = torch.rad2deg(torch.asin(dots))
    msd_perp = perp_deg.pow(2).mean(dim=1)
    rms_perp = msd_perp.sqrt()

    # Evenness: coefficient of variation of inter-star spacings
    cos_dots = (sorted_uv[:, :-1] * sorted_uv[:, 1:]).sum(dim=2).clamp(-1, 1)
    spacings = torch.acos(cos_dots)
    mean_sp = spacings.mean(dim=1)
    std_sp = spacings.std(dim=1)
    cv = torch.where(mean_sp > 1e-12, std_sp / mean_sp, torch.zeros_like(mean_sp))

    scores = {
        'rms': rms_perp + 0.3 * cv,
        'msd': msd_perp + 0.3 * cv,
    }

    # Smoothness: std of turning angles between consecutive segments
    dirs = sorted_uv[:, 1:] - sorted_uv[:, :-1]  # (N, K-1, 3)
    dir_norms = torch.linalg.norm(dirs, dim=2, keepdim=True).clamp(min=1e-12)
    dirs_normed = dirs / dir_norms
    cos_turn = (dirs_normed[:, :-1] * dirs_normed[:, 1:]).sum(dim=2).clamp(-1, 1)  # (N, K-2)
    turn_angles_deg = torch.rad2deg(torch.acos(cos_turn))
    turn_std = turn_angles_deg.std(dim=1)

    if chains_mag is not None:
        sorted_mag = chains_mag[batch_idx, sort_order]
        mean_mag = sorted_mag.mean(dim=1)
        std_mag = sorted_mag.std(dim=1)
        mag_cv = torch.where(mean_mag > 1e-12, std_mag / mean_mag, torch.zeros_like(mean_mag))
        scores['smooth'] = turn_std + 0.2 * cv + 0.3 * mag_cv
        # Magnitude gradient smoothness: std of consecutive magnitude differences
        mag_deltas = sorted_mag[:, 1:] - sorted_mag[:, :-1]  # (N, K-1)
        mag_gradient_std = mag_deltas.std(dim=1)
        scores['smooth_mag'] = turn_std + 0.2 * cv + 0.2 * mag_gradient_std
    else:
        scores['smooth'] = turn_std + 0.2 * cv
        scores['smooth_mag'] = turn_std + 0.2 * cv

    # Snake scoring: use ORIGINAL discovery order (not SVD-sorted) to preserve
    # S-curve topology. SVD sorting scrambles curves by projecting onto principal axis.
    if K >= 5:
        snake_dirs = chains_uv[:, 1:] - chains_uv[:, :-1]  # original order
        snake_dir_norms = torch.linalg.norm(snake_dirs, dim=2, keepdim=True).clamp(min=1e-12)
        snake_dirs_n = snake_dirs / snake_dir_norms
        snake_cos_turn = (snake_dirs_n[:, :-1] * snake_dirs_n[:, 1:]).sum(dim=2).clamp(-1, 1)
        snake_turn_deg = torch.rad2deg(torch.acos(snake_cos_turn))

        # Spacing CV from original order
        snake_spacings = torch.acos((chains_uv[:, :-1] * chains_uv[:, 1:]).sum(dim=2).clamp(-1, 1))
        snake_mean_sp = snake_spacings.mean(dim=1)
        snake_std_sp = snake_spacings.std(dim=1)
        snake_cv = torch.where(snake_mean_sp > 1e-12, snake_std_sp / snake_mean_sp, torch.zeros_like(snake_mean_sp))

        # Alternation: cross products in original order
        crosses = torch.cross(snake_dirs_n[:, :-1], snake_dirs_n[:, 1:], dim=2)
        chain_normal = Vh[:, 2, :]  # (N, 3) - normal to best-fit plane
        turn_sign = (crosses * chain_normal.unsqueeze(1)).sum(dim=2)
        signs = torch.sign(turn_sign)
        sign_changes = (signs[:, :-1] * signs[:, 1:] < 0).float()
        alternation_ratio = sign_changes.mean(dim=1)  # 1.0 = perfect snake

        # Turn amplitude regularity
        turn_abs = snake_turn_deg.abs()
        turn_mean = turn_abs.mean(dim=1)
        turn_angle_cv = torch.where(turn_mean > 1e-12,
                                     turn_abs.std(dim=1) / turn_mean,
                                     torch.zeros_like(turn_mean))

        # Penalize large mean turn angles (gentle S-curves preferred over zigzags)
        turn_mag_penalty = ((turn_mean - 15.0) / 30.0).clamp(min=0.0)
        # Hard filter: mean turn > 45° is zigzag, not snake
        too_zigzag = turn_mean > 45.0

        scores['snake'] = (1.0 - alternation_ratio) + 0.3 * turn_angle_cv + 0.2 * snake_cv + 0.4 * turn_mag_penalty
        scores['snake'] = torch.where(too_zigzag, torch.tensor(999.0, device=device), scores['snake'])
    else:
        # Not enough turns for meaningful alternation — assign worst score
        scores['snake'] = torch.full((N,), 999.0, device=device)

    return scores, sort_order
def _chain_extent_batch(sorted_stars):
    """Vectorized angular extent for sorted chains (endpoint separation)."""
    ra0 = torch.deg2rad(sorted_stars[:, 0, 0])
    dec0 = torch.deg2rad(sorted_stars[:, 0, 1])
    ra1 = torch.deg2rad(sorted_stars[:, -1, 0])
    dec1 = torch.deg2rad(sorted_stars[:, -1, 1])
    cos_sep = (torch.sin(dec0) * torch.sin(dec1) +
               torch.cos(dec0) * torch.cos(dec1) * torch.cos(ra0 - ra1)).clamp(-1, 1)
    return torch.rad2deg(torch.acos(cos_sep))


def _score_chain(chain_uv):
    """Score a single chain's straightness + evenness. Returns (score, sorted_order)."""
    K = len(chain_uv)
    centroid = chain_uv.mean(dim=0)
    centered = chain_uv - centroid.unsqueeze(0)
    _, _, Vh = torch.linalg.svd(centered, full_matrices=False)
    projections = chain_uv @ Vh[0]
    sort_order = torch.argsort(projections)
    sorted_uv = chain_uv[sort_order]

    # Straightness
    normal = torch.cross(sorted_uv[0], sorted_uv[-1], dim=0)
    norm = torch.linalg.norm(normal)
    if norm < 1e-12:
        rms_perp = 0.0
    else:
        normal = normal / norm
        interior_idx = list(range(1, K - 1))
        if interior_idx:
            dots = torch.abs(sorted_uv[interior_idx] @ normal).clamp(0, 1)
            perp_deg = torch.rad2deg(torch.asin(dots))
            rms_perp = perp_deg.pow(2).mean().sqrt().item()
        else:
            rms_perp = 0.0

    # Evenness
    cos_dots = (sorted_uv[:-1] * sorted_uv[1:]).sum(dim=1).clamp(-1, 1)
    spacings = torch.acos(cos_dots)
    mean_sp = spacings.mean()
    cv = (spacings.std() / mean_sp).item() if mean_sp > 1e-12 else 0.0

    return rms_perp + 0.3 * cv, sort_order


def score_collinear_region(stars_tensor, device='cpu', mode='2d', angle_tol_deg=0.5):
    """Single-region collinear search (fallback for CPU path)."""
    stars_tensor = stars_tensor.to(device)
    N = len(stars_tensor)
    if N < 4:
        return []

    padded = stars_tensor.unsqueeze(0)  # (1, N, 3)
    counts = torch.tensor([N], dtype=torch.long)
    sa, si, unit_vecs = _batch_compute_angles(padded, counts, device)
    candidates = _find_chains_batch(sa, si, counts.numpy(), math.radians(angle_tol_deg))

    uv = unit_vecs[0]  # (N, 3)
    scored = []
    for _, chain_idx in candidates:
        idx_list = list(chain_idx)
        chain_uv = uv[idx_list]
        score, sort_order = _score_chain(chain_uv)
        chain_stars = stars_tensor[idx_list][sort_order]
        scored.append((score, chain_stars))

    scored.sort(key=lambda x: x[0])
    return scored[:5]



def process_collinear_regions(grid_points, gpu_catalog, config, device, max_extent=None):
    """Process ALL regions for collinear/smooth chains in batched GPU passes.
    
    Phase 1: Filter regions on GPU
    Phase 2a: KNN-bounded angle matrices on GPU + numpy chain finding (straight lines)
    Phase 2b: Greedy smooth chain finding (curves)
    Phase 3: Batch score all candidates on GPU, grouped by chain length
    
    Memory-bounded: stores stars as numpy arrays (not Python lists) and
    periodically prunes accumulated results to top-50 per region/chain_len.
    """
    radius = config.search_radius_deg
    max_mag = config.max_mag
    max_stars = config.max_stars_per_region if config.max_stars_per_region > 0 else 200
    angle_tol_rad = math.radians(0.5)

    # --- Phase 1: Filter all regions, build padded tensor ---
    print("Phase 1: Filtering regions...")
    region_stars = []  # (grid_idx, stars_tensor)
    for idx, point in tqdm.tqdm(grid_points, desc="Filter"):
        ra, dec = point
        mask = (
            (gpu_catalog[:, 0] >= ra) & (gpu_catalog[:, 0] < ra + radius) &
            (gpu_catalog[:, 1] >= dec) & (gpu_catalog[:, 1] < dec + radius) &
            (gpu_catalog[:, 2] <= max_mag)
        )
        stars = gpu_catalog[mask]
        if len(stars) >= 4:
            if max_stars > 0 and len(stars) > max_stars:
                _, top_idx = torch.topk(stars[:, 2], max_stars, largest=False)
                stars = stars[top_idx]
            # Dedup stars with near-identical coordinates (within ~0.36 arcsec)
            coords_rounded = torch.round(stars[:, :2] * 10000)
            _, inv = torch.unique(coords_rounded, dim=0, return_inverse=True)
            n_unique = inv.max().item() + 1
            first_occ = torch.full((n_unique,), len(stars), dtype=torch.long, device=stars.device)
            first_occ.scatter_reduce_(0, inv, torch.arange(len(stars), device=stars.device), reduce='amin')
            stars = stars[first_occ]
            if len(stars) < 4:
                continue
            region_stars.append((idx, stars))

    if not region_stars:
        return pl.DataFrame()

    B = len(region_stars)
    actual_max_n = max(len(s) for _, s in region_stars)
    print(f"Phase 1 done: {B} regions with stars (max {actual_max_n}/region)")

    # Build padded tensors for star data and grid indices
    concat_stars = torch.zeros(B, actual_max_n, 3, device=device)
    grid_indices = torch.zeros(B, dtype=torch.long, device=device)
    region_counts = torch.zeros(B, dtype=torch.long)
    for b, (idx, stars) in enumerate(region_stars):
        concat_stars[b, :len(stars)] = stars
        grid_indices[b] = idx
        region_counts[b] = len(stars)
    del region_stars
    torch.cuda.empty_cache()
    gc.collect()

    # --- Phases 2+3: Process in region chunks to limit memory ---
    # KNN-bounded angles use O(B*N*K) instead of O(B*N²).
    # Per-region KNN step uses O(N²) peak but processes one region at a time.
    # Budget: ~1GB for angle tensors + KNN gather
    k_angle = min(50, actual_max_n - 1)
    angle_mem_per_region = actual_max_n * k_angle * 4 * 8  # tensors during angle computation
    sub_batch_size = min(B, max(50, 1_000_000_000 // max(angle_mem_per_region, 1)))
    print(f"Phases 2+3: Processing in chunks of {sub_batch_size} regions (N={actual_max_n}, K={k_angle})...")

    score_keys = ['rms', 'msd', 'smooth', 'smooth_mag', 'snake']
    all_results = {k: [] for k in score_keys}
    all_results_regions = []
    all_results_lens = []
    all_results_stars = []  # list of numpy arrays (K, 3) per chain
    total_straight = 0
    total_smooth = 0
    total_scored = 0
    chunks_since_prune = 0
    PRUNE_INTERVAL = 5

    def _prune_accumulated():
        """Compact accumulated results, keeping top-20 per region/chain_len.

        Uses min rank across all scoring modes as proxy.  Final pass
        keeps top-10, so 20 provides headroom for overlapping regions.
        """
        if not all_results_regions:
            return
        regions = np.concatenate(all_results_regions)
        lens = np.concatenate(all_results_lens)
        scores = {k: np.concatenate(all_results[k]) for k in score_keys}
        n_total = len(regions)
        if n_total < 50_000:
            return

        # Best rank across all scoring modes
        best_rank = np.full(n_total, n_total, dtype=np.int64)
        for k in score_keys:
            cutoff = SCORE_CUTOFFS.get(k, 1.0)
            valid = scores[k] <= cutoff
            order = np.argsort(scores[k])
            rank = np.empty(n_total, dtype=np.int64)
            rank[order] = np.arange(n_total)
            rank[~valid] = n_total
            np.minimum(best_rank, rank, out=best_rank)

        # Keep top-20 per (region, chain_len)
        TOP_K = 20
        keep = np.zeros(n_total, dtype=bool)
        keys = (regions.astype(np.int64) << 16) | lens.astype(np.int64)
        for ukey in np.unique(keys):
            mask = keys == ukey
            idxs = np.where(mask)[0]
            if len(idxs) <= TOP_K:
                keep[idxs] = True
            else:
                topk = idxs[np.argpartition(best_rank[idxs], TOP_K)[:TOP_K]]
                keep[topk] = True

        n_kept = keep.sum()
        if n_kept >= n_total * 0.9:
            return

        all_results_regions.clear()
        all_results_regions.append(regions[keep])
        all_results_lens.clear()
        all_results_lens.append(lens[keep])
        for k in score_keys:
            all_results[k].clear()
            all_results[k].append(scores[k][keep])

        old_stars = all_results_stars.copy()
        all_results_stars.clear()
        for i in np.where(keep)[0]:
            all_results_stars.append(old_stars[i])
        del old_stars

        print(f"  Pruned {n_total:,} -> {n_kept:,} chains ({100*n_kept/n_total:.0f}%)")
        gc.collect()

    for sb_start in tqdm.tqdm(range(0, B, sub_batch_size), desc="Chunks"):
        sb_end = min(sb_start + sub_batch_size, B)
        sb_B = sb_end - sb_start

        # Phase 2a: KNN-bounded angle computation + straight chain finding
        padded = concat_stars[sb_start:sb_end].clone()
        counts = region_counts[sb_start:sb_end].clone()

        sa, si, unit_vecs = _batch_compute_angles(padded, counts, device, k_angle=k_angle)
        candidates = _find_chains_batch(sa, si, counts.numpy(), angle_tol_rad)
        del sa, si

        chunk_candidates = []
        for b_local, chain_idx in candidates:
            global_b = sb_start + b_local
            chunk_candidates.append((global_b, b_local, chain_idx))
        del candidates
        total_straight += len(chunk_candidates)

        # Phase 2b: smooth chain finding for this chunk
        seen = set()
        for _, _, chain_idx in chunk_candidates:
            seen.add(frozenset(chain_idx))

        for b_local in range(sb_B):
            n = int(counts[b_local])
            if n < 4:
                continue
            uv_np = unit_vecs[b_local, :n].cpu().numpy()
            smooth_chains = _find_smooth_chains(uv_np, max_turn_deg=30.0, min_stars=4, k_neighbors=15)
            global_b = sb_start + b_local
            for chain_idx in smooth_chains:
                key = frozenset(chain_idx)
                if key not in seen:
                    seen.add(key)
                    chunk_candidates.append((global_b, b_local, chain_idx))
                    total_smooth += 1
        del seen

        if not chunk_candidates:
            del padded, unit_vecs
            torch.cuda.empty_cache()
            gc.collect()
            continue

        # Phase 3: Score this chunk's candidates on GPU
        by_length = defaultdict(list)
        for global_b, b_local, chain_idx in chunk_candidates:
            by_length[len(chain_idx)].append((global_b, b_local, list(chain_idx)))
        del chunk_candidates

        for K in sorted(by_length.keys()):
            group = by_length[K]
            N_K = len(group)

            score_batch_size = 500_000
            for g_start in range(0, N_K, score_batch_size):
                g_end = min(g_start + score_batch_size, N_K)
                batch = group[g_start:g_end]
                n_batch = len(batch)

                gb = torch.tensor([g[0] for g in batch], device=device, dtype=torch.long)
                gb_local = torch.tensor([g[1] for g in batch], device=device, dtype=torch.long)
                ci = torch.tensor([g[2] for g in batch], device=device, dtype=torch.long)

                gb_exp = gb.unsqueeze(1).expand(-1, K)
                gb_local_exp = gb_local.unsqueeze(1).expand(-1, K)
                uv_batch = unit_vecs[gb_local_exp, ci]  # chunk-local indexing
                stars_batch = concat_stars[gb_exp, ci]   # global indexing
                mag_batch = concat_stars[gb_exp, ci, 2]

                scores_dict, sort_orders = _batch_score_chains(uv_batch, chains_mag=mag_batch, max_mag=max_mag)

                # SVD-sorted stars for extent filtering (endpoints are physical extremes)
                bidx = torch.arange(n_batch, device=device).unsqueeze(1).expand(-1, K)
                sorted_stars = stars_batch[bidx, sort_orders]

                if max_extent is not None:
                    extents = _chain_extent_batch(sorted_stars)
                    mask = extents <= max_extent
                    for k in score_keys:
                        scores_dict[k] = scores_dict[k][mask]
                    sorted_stars = sorted_stars[mask]
                    stars_batch = stars_batch[mask]
                    gb = gb[mask]

                n_kept = len(scores_dict['rms'])
                if n_kept > 0:
                    for k in score_keys:
                        all_results[k].append(scores_dict[k].cpu().numpy())
                    all_results_regions.append(grid_indices[gb].cpu().numpy())
                    all_results_lens.append(np.full(n_kept, K, dtype=np.int32))
                    # Save in discovery order as numpy — preserves spatial path
                    # for smooth/snake chains. SVD sort destroys S-curve topology.
                    stars_cpu = stars_batch.cpu().numpy()
                    for row in range(n_kept):
                        all_results_stars.append(stars_cpu[row])
                    total_scored += n_kept

                del uv_batch, stars_batch, scores_dict, sort_orders, sorted_stars, gb, gb_local, ci, mag_batch, gb_local_exp
                torch.cuda.empty_cache()

                # Mid-chunk RSS check
                if len(all_results_stars) > 100_000:
                    rss_gb = _get_rss_gb()
                    if rss_gb > 15:
                        _prune_accumulated()
                        gc.collect()
                        chunks_since_prune = 0

        del by_length, padded, unit_vecs
        torch.cuda.empty_cache()
        gc.collect()

        # Periodically prune to bound memory
        chunks_since_prune += 1
        if chunks_since_prune >= PRUNE_INTERVAL:
            _prune_accumulated()
            chunks_since_prune = 0

        # RSS-based emergency prune (prevent OOM on dense regions)
        rss_gb = _get_rss_gb()
        if rss_gb > 20:
            print(f"  RSS {rss_gb:.1f} GB — emergency prune")
            _prune_accumulated()
            gc.collect()
            chunks_since_prune = 0

    del concat_stars
    print(f"All chunks done: {total_straight:,} straight + {total_smooth:,} smooth candidates, {total_scored:,} scored")

    if not all_results_regions:
        print("Phase 3 done: 0 chains after extent filter")
        return pl.DataFrame()

    regions_arr = np.concatenate(all_results_regions)
    lens_arr = np.concatenate(all_results_lens)

    print(f"Phase 3 done: {len(regions_arr):,} chains after extent filter")

    # Convert stars from numpy arrays to Polars list column
    stars_as_lists = [s.tolist() for s in all_results_stars]
    del all_results_stars

    result = pl.DataFrame({
        "score_rms": np.concatenate(all_results['rms']).astype(np.float64),
        "score_msd": np.concatenate(all_results['msd']).astype(np.float64),
        "score_smooth": np.concatenate(all_results['smooth']).astype(np.float64),
        "score_smooth_mag": np.concatenate(all_results['smooth_mag']).astype(np.float64),
        "score_snake": np.concatenate(all_results['snake']).astype(np.float64),
        "region": regions_arr.astype(np.int64),
        "chain_len": lens_arr.astype(np.int32),
        "stars": stars_as_lists,
    })
    del stars_as_lists

    # Keep top 10 per region per chain_len, independently per scoring mode
    results = {}
    for k in score_keys:
        score_col = f"score_{k}"
        cutoff = SCORE_CUTOFFS.get(k, 1.0)
        df_k = result.select([
            pl.col(score_col).alias("score"),
            pl.col("region"), pl.col("chain_len"), pl.col("stars"),
        ]).filter(pl.col("score") <= cutoff)
        df_k = df_k.sort("score").group_by(["region", "chain_len"]).head(10)
        results[k] = df_k

    total = sum(len(v) for v in results.values())
    print(f"After top-10 filter + cutoffs: {total:,} chains (across {len(score_keys)} scoring modes)")

    return results


## Bright isolated lines and circle/arc detection

In [ ]:
#| export


def compute_chain_corridor_flux(chain_stars, catalog_df, max_mag, corridor_mult=2.0):
    """Compute flux ratio of chain stars vs all stars in a corridor around the chain.

    Args:
        chain_stars: list of [RA, Dec, Vmag] for chain stars
        catalog_df: Polars DataFrame with RAmdeg, DEmdeg, Vmag columns
        max_mag: limiting magnitude
        corridor_mult: corridor width as multiple of mean inter-star spacing

    Returns:
        flux_ratio: chain flux / total corridor flux (1.0 = chain dominates)
        mag_contrast: median(field_mags) - median(chain_mags) (positive = chain brighter)
    """
    pts = np.array(chain_stars)
    chain_ra = pts[:, 0]
    chain_dec = pts[:, 1]
    chain_mag = pts[:, 2]

    # Mean inter-star spacing (angular)
    diffs = np.diff(pts[:, :2], axis=0)
    # Correct RA spacing for declination
    cos_dec = np.cos(np.radians(np.mean(chain_dec)))
    diffs[:, 0] *= cos_dec
    spacings = np.sqrt((diffs ** 2).sum(axis=1))
    mean_spacing = spacings.mean() if len(spacings) > 0 else 0.1

    corridor_width = mean_spacing * corridor_mult

    # Bounding box of chain expanded by corridor
    ra_min = chain_ra.min() - corridor_width / cos_dec
    ra_max = chain_ra.max() + corridor_width / cos_dec
    dec_min = chain_dec.min() - corridor_width
    dec_max = chain_dec.max() + corridor_width

    # Query catalog for corridor stars
    corridor = catalog_df.filter(
        (pl.col("RAmdeg") >= ra_min) & (pl.col("RAmdeg") <= ra_max) &
        (pl.col("DEmdeg") >= dec_min) & (pl.col("DEmdeg") <= dec_max) &
        (pl.col("Vmag") <= max_mag)
    )

    # Flux calculations
    flux_chain = np.sum(10 ** (-0.4 * chain_mag))
    if len(corridor) == 0:
        return 1.0, 0.0

    field_mags = corridor["Vmag"].to_numpy()
    flux_field = np.sum(10 ** (-0.4 * field_mags))

    flux_ratio = flux_chain / flux_field if flux_field > 0 else 1.0
    mag_contrast = float(np.median(field_mags) - np.median(chain_mag))

    return float(flux_ratio), mag_contrast


def score_bright_isolated_chains(collinear_df, catalog_df, max_mag):
    """Re-score collinear chains by brightness contrast against surrounding field.

    Args:
        collinear_df: DataFrame with 'score', 'stars', 'chain_len', 'region' columns
        catalog_df: full star catalog DataFrame
        max_mag: limiting magnitude

    Returns:
        DataFrame with 'bright_score', 'flux_ratio', 'mag_contrast' columns added
    """
    bright_scores = []
    flux_ratios = []
    mag_contrasts = []

    for row in collinear_df.iter_rows(named=True):
        chain_stars = row['stars']
        flux_ratio, mag_contrast = compute_chain_corridor_flux(chain_stars, catalog_df, max_mag)
        # Lower bright_score = better (chain dominates field and has good geometry)
        geometric_score = row['score']
        bright_score = geometric_score / (flux_ratio * (1.0 + 0.5 * max(mag_contrast, 0.0)))
        bright_scores.append(bright_score)
        flux_ratios.append(flux_ratio)
        mag_contrasts.append(mag_contrast)

    return collinear_df.with_columns([
        pl.Series("bright_score", bright_scores),
        pl.Series("flux_ratio", flux_ratios),
        pl.Series("mag_contrast", mag_contrasts),
    ])


def _gnomonic_project(stars_radec, center_ra, center_dec):
    """Project RA/Dec (degrees) to gnomonic tangent plane (x, y) in degrees.

    Args:
        stars_radec: (N, 2) tensor or array of [RA, Dec] in degrees
        center_ra, center_dec: tangent point in degrees

    Returns:
        xy: (N, 2) tensor of tangent plane coordinates in degrees
    """
    ra = torch.deg2rad(stars_radec[:, 0])
    dec = torch.deg2rad(stars_radec[:, 1])
    ra0 = math.radians(center_ra)
    dec0 = math.radians(center_dec)

    cos_dec = torch.cos(dec)
    sin_dec = torch.sin(dec)
    cos_dec0 = math.cos(dec0)
    sin_dec0 = math.sin(dec0)
    delta_ra = ra - ra0

    cos_c = sin_dec0 * sin_dec + cos_dec0 * cos_dec * torch.cos(delta_ra)
    cos_c = cos_c.clamp(min=1e-12)

    x = cos_dec * torch.sin(delta_ra) / cos_c
    y = (cos_dec0 * sin_dec - sin_dec0 * cos_dec * torch.cos(delta_ra)) / cos_c

    return torch.rad2deg(torch.stack([x, y], dim=1))


def _circumcircle_batch(xy, triplet_indices):
    """Compute circumscribed circle for batches of triangles in 2D.

    Args:
        xy: (N, 2) tensor of 2D point coordinates
        triplet_indices: (M, 3) tensor of point indices

    Returns:
        centers: (M, 2) tensor of circle centers
        radii: (M,) tensor of circle radii
        valid: (M,) boolean tensor (False for degenerate triangles)
    """
    A = xy[triplet_indices[:, 0]]  # (M, 2)
    B = xy[triplet_indices[:, 1]]
    C = xy[triplet_indices[:, 2]]

    D = 2.0 * (A[:, 0] * (B[:, 1] - C[:, 1]) +
               B[:, 0] * (C[:, 1] - A[:, 1]) +
               C[:, 0] * (A[:, 1] - B[:, 1]))

    valid = D.abs() > 1e-12

    # Avoid division by zero for degenerate triangles
    D_safe = torch.where(valid, D, torch.ones_like(D))

    A_sq = (A ** 2).sum(dim=1)
    B_sq = (B ** 2).sum(dim=1)
    C_sq = (C ** 2).sum(dim=1)

    cx = (A_sq * (B[:, 1] - C[:, 1]) +
          B_sq * (C[:, 1] - A[:, 1]) +
          C_sq * (A[:, 1] - B[:, 1])) / D_safe
    cy = (A_sq * (C[:, 0] - B[:, 0]) +
          B_sq * (A[:, 0] - C[:, 0]) +
          C_sq * (B[:, 0] - A[:, 0])) / D_safe

    centers = torch.stack([cx, cy], dim=1)
    radii = torch.sqrt((A[:, 0] - cx) ** 2 + (A[:, 1] - cy) ** 2)

    return centers, radii, valid


def _count_stars_on_circles(xy, centers, radii, tolerance_deg=0.005, batch_size=100_000,
                           min_count=4):
    """Count stars within tolerance of each circle, return only circles with >= min_count.

    Uses squared-distance comparison to avoid expensive sqrt in the hot loop.

    Args:
        xy: (N, 2) tangent plane coordinates
        centers: (M, 2) circle centers
        radii: (M,) circle radii
        tolerance_deg: absolute tolerance in degrees
        batch_size: process circles in chunks to limit memory
        min_count: minimum stars to keep a circle

    Returns:
        good_centers: (G, 2) centers of good circles
        good_radii: (G,) radii of good circles
        good_star_indices: list of G tensors of star indices
    """
    M = len(centers)
    device = xy.device

    # Phase 1: Count on GPU using squared distances (no sqrt needed)
    all_counts = torch.zeros(M, dtype=torch.long, device=device)

    for b_start in range(0, M, batch_size):
        b_end = min(b_start + batch_size, M)
        b_centers = centers[b_start:b_end]
        b_radii = radii[b_start:b_end]

        dx = xy[:, 0].unsqueeze(0) - b_centers[:, 0].unsqueeze(1)
        dy = xy[:, 1].unsqueeze(0) - b_centers[:, 1].unsqueeze(1)
        dist_sq = dx ** 2 + dy ** 2

        tol = torch.clamp(0.01 * b_radii, min=tolerance_deg)
        r_lo_sq = (b_radii - tol).clamp(min=0).unsqueeze(1) ** 2
        r_hi_sq = (b_radii + tol).unsqueeze(1) ** 2
        all_counts[b_start:b_end] = ((dist_sq >= r_lo_sq) & (dist_sq <= r_hi_sq)).sum(dim=1)

        del dx, dy, dist_sq

    # Phase 2: Collect indices only for good circles (much smaller set)
    good_mask = all_counts >= min_count
    good_idx = torch.where(good_mask)[0]

    if len(good_idx) == 0:
        return centers[:0], radii[:0], []

    # Limit to top 500 by count for speed
    if len(good_idx) > 500:
        good_counts = all_counts[good_idx]
        _, top_k = torch.topk(good_counts, 500)
        good_idx = good_idx[top_k]

    good_centers = centers[good_idx]
    good_radii = radii[good_idx]

    # Collect star indices: batch nonzero + split by row counts
    dx = xy[:, 0].unsqueeze(0) - good_centers[:, 0].unsqueeze(1)  # (G, N)
    dy = xy[:, 1].unsqueeze(0) - good_centers[:, 1].unsqueeze(1)
    dists = torch.sqrt(dx ** 2 + dy ** 2)
    tol = torch.clamp(0.01 * good_radii, min=tolerance_deg).unsqueeze(1)
    on_circle = torch.abs(dists - good_radii.unsqueeze(1)) < tol  # (G, N) bool
    row_counts = on_circle.sum(dim=1)
    nz = torch.nonzero(on_circle)
    if len(nz) > 0:
        good_star_indices = list(torch.split(nz[:, 1], row_counts.tolist()))
    else:
        good_star_indices = [torch.tensor([], dtype=torch.long, device=device)] * len(good_idx)
    del dx, dy, dists

    return good_centers, good_radii, good_star_indices


def _score_circle_candidates(xy, mags, candidates, search_radius_deg):
    """Score circle/arc candidates.

    Args:
        xy: (N, 2) tangent plane coordinates
        mags: (N,) magnitudes
        candidates: list of dicts with 'center', 'radius', 'star_indices'
        search_radius_deg: max search radius for filtering

    Returns:
        scored: list of dicts with added 'score', 'arc_fraction', 'n_stars'
    """
    scored = []
    for cand in candidates:
        star_idx = cand['star_indices']
        if len(star_idx) < 4:
            continue

        center = cand['center']
        radius = cand['radius']
        pts = xy[star_idx]
        star_mags = mags[star_idx]

        # Radial deviation (normalized by radius)
        dists = torch.sqrt(((pts - center) ** 2).sum(dim=1))
        radial_devs = (dists - radius).abs() / radius
        rms_radial = radial_devs.pow(2).mean().sqrt().item()

        # Angular positions around circle
        dx = pts[:, 0] - center[0]
        dy = pts[:, 1] - center[1]
        angles = torch.atan2(dy, dx)
        sorted_angles, sort_idx = torch.sort(angles)

        # Angular gaps between consecutive stars
        gaps = sorted_angles[1:] - sorted_angles[:-1]
        # Wrap-around gap
        wrap_gap = (2 * math.pi) - (sorted_angles[-1] - sorted_angles[0])
        all_gaps = torch.cat([gaps, wrap_gap.unsqueeze(0)])
        gap_mean = all_gaps.mean()
        gap_std = all_gaps.std()
        angular_spacing_cv = (gap_std / gap_mean).item() if gap_mean > 1e-12 else 0.0

        # Arc fraction: angular span / 2pi
        arc_span = (sorted_angles[-1] - sorted_angles[0]).item()
        if arc_span < 0:
            arc_span += 2 * math.pi
        arc_fraction = arc_span / (2 * math.pi)

        # Combined score (lower = better)
        # More stars allow more radial scatter — n_bonus rewards larger circles
        n = len(star_idx)
        n_bonus = math.log2(n / 4) if n > 4 else 0.0
        raw_score = rms_radial + 0.3 * angular_spacing_cv + 0.3 * (1.0 - arc_fraction)
        score = raw_score / (1.0 + 0.5 * n_bonus)

        scored.append({
            'score': score,
            'center': center.cpu().numpy().tolist(),
            'radius': radius,
            'star_indices': star_idx,
            'stars': torch.stack([
                xy[star_idx[sort_idx], 0],
                xy[star_idx[sort_idx], 1],
                star_mags[sort_idx],
            ], dim=1),
            'n_stars': len(star_idx),
            'arc_fraction': arc_fraction,
            'radius_deg': radius,
        })

    return scored


def _dedup_circles(candidates, merge_radius_frac=0.3):
    """Cluster nearby circle centers and merge star sets.

    Keeps the candidate with the best (lowest) score from each cluster.
    """
    if not candidates:
        return []

    # Sort by score
    candidates.sort(key=lambda c: c['score'])
    kept = []
    used = set()

    for i, cand in enumerate(candidates):
        if i in used:
            continue
        kept.append(cand)
        ci = np.array(cand['center'])
        ri = cand['radius_deg']
        for j in range(i + 1, len(candidates)):
            if j in used:
                continue
            cj = np.array(candidates[j]['center'])
            rj = candidates[j]['radius_deg']
            dist = np.sqrt(((ci - cj) ** 2).sum())
            # Merge if centers are close relative to radii
            if dist < merge_radius_frac * (ri + rj):
                used.add(j)

    return kept


def process_circle_regions(grid_points, gpu_catalog, config, device='cuda',
                           max_extent=None, min_stars_on_circle=5):
    """Find circles/arcs of stars across sky regions.

    Uses GPU catalog tensor for fast region filtering (like collinear pipeline).
    Optimized with squared-distance star counting and torch.combinations for
    triplet generation.

    Args:
        grid_points: list of (idx, (ra, dec)) grid points
        gpu_catalog: PyTorch tensor (N, 3+) on GPU with [RA, Dec, Vmag, ...]
        config: SearchConfig
        device: torch device
        max_extent: max angular extent filter
        min_stars_on_circle: minimum stars to form a circle candidate

    Returns:
        Polars DataFrame with score, region, stars, n_stars, radius_deg, arc_fraction
    """
    radius = config.search_radius_deg
    max_mag = config.max_mag
    max_stars = config.max_stars_per_region if config.max_stars_per_region else 200
    min_radius = 0.02  # 1 arcmin minimum
    max_radius = radius * 0.8

    # Phase 1: GPU-based region filtering
    print("Phase 1: Filtering regions on GPU...")
    region_stars = []
    for idx, point in tqdm.tqdm(grid_points, desc="Filter"):
        ra, dec = point
        mask = (
            (gpu_catalog[:, 0] >= ra) & (gpu_catalog[:, 0] < ra + radius) &
            (gpu_catalog[:, 1] >= dec) & (gpu_catalog[:, 1] < dec + radius) &
            (gpu_catalog[:, 2] <= max_mag)
        )
        stars = gpu_catalog[mask]
        if len(stars) >= min_stars_on_circle:
            if max_stars > 0 and len(stars) > max_stars:
                _, top_idx = torch.topk(stars[:, 2], max_stars, largest=False)
                stars = stars[top_idx]
            # Dedup near-identical coordinates
            coords_rounded = torch.round(stars[:, :2] * 10000)
            _, inv = torch.unique(coords_rounded, dim=0, return_inverse=True)
            n_unique = inv.max().item() + 1
            first_occ = torch.full((n_unique,), len(stars), dtype=torch.long, device=stars.device)
            first_occ.scatter_reduce_(0, inv, torch.arange(len(stars), device=stars.device), reduce='amin')
            stars = stars[first_occ]
            if len(stars) >= min_stars_on_circle:
                region_stars.append((idx, stars))

    print(f"  {len(region_stars):,} regions with >= {min_stars_on_circle} stars")

    if not region_stars:
        return pl.DataFrame()

    # Phase 2: Circle detection per region
    all_results = []
    n_regions = len(region_stars)

    triplet_cache = {}
    for reg_i, (idx, stars_t) in enumerate(region_stars):
        if reg_i % 1000 == 0:
            print(f"  Circle region {reg_i}/{n_regions}...")

        N = len(stars_t)
        ra_center = stars_t[:, 0].mean().item()
        dec_center = stars_t[:, 1].mean().item()

        # Project to tangent plane
        xy = _gnomonic_project(stars_t[:, :2], ra_center, dec_center)
        mags = stars_t[:, 2]

        # Generate triplets on GPU (cached per N)
        if N > 200:
            # Random subsample for very dense regions
            n_triplets = 1_300_000
            idx_a = torch.randint(0, N, (n_triplets,), device=device)
            idx_b = torch.randint(0, N, (n_triplets,), device=device)
            idx_c = torch.randint(0, N, (n_triplets,), device=device)
            valid_mask = (idx_a != idx_b) & (idx_b != idx_c) & (idx_a != idx_c)
            triplet_indices = torch.stack([idx_a[valid_mask], idx_b[valid_mask], idx_c[valid_mask]], dim=1)
        else:
            if N not in triplet_cache:
                triplet_cache[N] = torch.combinations(torch.arange(N, device=device), r=3)
            triplet_indices = triplet_cache[N]

        if len(triplet_indices) == 0:
            continue

        # Compute circumcircles
        centers, radii_t, valid = _circumcircle_batch(xy, triplet_indices)

        # Filter by radius range and validity
        radius_mask = valid & (radii_t >= min_radius) & (radii_t <= max_radius)
        if radius_mask.sum() == 0:
            del xy, mags, triplet_indices, centers, radii_t
            torch.cuda.empty_cache()
            continue

        centers = centers[radius_mask]
        radii_t = radii_t[radius_mask]

        # Quantize circles to reduce duplicates before expensive counting
        quant = 0.01  # degrees (~36 arcsec)
        keys = torch.round(torch.cat([centers, radii_t.unsqueeze(1)], dim=1) / quant)
        _, inv = torch.unique(keys, dim=0, return_inverse=True)
        n_uniq = inv.max().item() + 1
        first_occ = torch.full((n_uniq,), len(centers), dtype=torch.long, device=device)
        first_occ.scatter_reduce_(0, inv, torch.arange(len(centers), device=device), reduce='amin')
        centers = centers[first_occ]
        radii_t = radii_t[first_occ]

        # Count stars on each circle, get only good ones
        good_centers, good_radii, good_star_indices = _count_stars_on_circles(
            xy, centers, radii_t, min_count=min_stars_on_circle)
        if len(good_star_indices) == 0:
            del xy, mags, triplet_indices, centers, radii_t
            torch.cuda.empty_cache()
            continue

        candidates = []
        for i in range(len(good_star_indices)):
            candidates.append({
                'center': good_centers[i],
                'radius': good_radii[i].item(),
                'star_indices': good_star_indices[i],
                'radius_deg': good_radii[i].item(),
            })

        # Score and dedup
        scored = _score_circle_candidates(xy, mags, candidates, radius)
        deduped = _dedup_circles(scored)

        # Keep top 5 per region
        for cand in deduped[:5]:
            star_idx = cand['star_indices']
            star_list = stars_t[star_idx].cpu().numpy().tolist()
            all_results.append({
                'score': cand['score'],
                'region': idx,
                'stars': star_list,
                'n_stars': cand['n_stars'],
                'radius_deg': cand['radius_deg'],
                'arc_fraction': cand['arc_fraction'],
            })

        del xy, mags, triplet_indices, centers, radii_t
        torch.cuda.empty_cache()

    if not all_results:
        return pl.DataFrame()

    return pl.DataFrame(all_results)


## Star filtering and grid utilities

In [7]:
#| export
def stars_for_point_and_radius(df, point, radius, max_mag):
    """ point is in the corner, not the center """
    ra, dec = point
    minra = ra
    maxra = ra + radius
    mindec = dec
    maxdec = dec + radius
    result = df.filter((pl.col("RAmdeg") < maxra) & (pl.col("RAmdeg") > minra) & (pl.col("DEmdeg") < maxdec) & (pl.col("DEmdeg") > mindec) & (pl.col("Vmag") <= max_mag))
    return result

def stars_for_center_and_radius(df, center, radius, max_mag):
    ra, dec = center
    minra = ra - radius/2
    maxra = ra + radius/2
    mindec = dec - radius/2
    maxdec = dec + radius/2
    return df.filter((pl.col("RAmdeg") < maxra) & (pl.col("RAmdeg") > minra) & (pl.col("DEmdeg") < maxdec) & (pl.col("DEmdeg") > mindec) & (pl.col("Vmag") <= max_mag))


def get_grid_points(config_or_step=None, min_dec=-90, max_dec=90):
    """Generate grid points for sky survey.
    
    Args:
        config_or_step: SearchConfig, int step size, or None (default 4deg)
    """
    if isinstance(config_or_step, SearchConfig):
        step = config_or_step.grid_step_deg
    elif isinstance(config_or_step, (int, float)):
        step = config_or_step
    elif config_or_step is None:
        step = 4
    else:
        step = float(config_or_step)
    RA_values = np.arange(0, 361, step).tolist()
    Dec_values = np.arange(min_dec, max_dec + step, step).tolist()
    grid_points = [(ra, dec) for ra in RA_values for dec in Dec_values]
    return grid_points

def get_grid_point_by_idx(idx, config=None):
    gp = get_grid_points(config)
    return gp[idx]

def get_region(df, idx, radius, max_mag, min_dec=-90, max_dec=90, config=None):
    center = get_grid_points(config, min_dec, max_dec)[idx]
    return stars_for_center_and_radius(df, center, radius, max_mag)

def get_center(dftycho, center, radius, max_mag):
    return stars_for_point_and_radius(dftycho, center, radius, max_mag)

def ra_to_hms(ra):
    if ra < 0.0:
        ra = ra + 360
    mm, hh = math.modf(ra / 15.0)
    _, mm = math.modf(mm * 60.0)
    ss = round(_ * 60.0)
    return hh, mm, ss

resultdf = pl.DataFrame({
    "score": pl.Float64,
    "region": pl.Int64,
    "item": []
})

## GPU pipeline

In [8]:
#| export

# Global GPU catalog cache
_gpu_catalog = None

def load_catalog_to_gpu(df, device='cuda'):
    """Load entire star catalog to GPU once for faster filtering"""
    global _gpu_catalog
    if _gpu_catalog is None and torch.cuda.is_available() and device == 'cuda':
        catalog_array = df.select(['RAmdeg', 'DEmdeg', 'Vmag']).to_numpy()
        _gpu_catalog = torch.tensor(catalog_array, dtype=torch.float32, device=device)
        print(f"Loaded {len(df):,} stars to GPU VRAM ({_gpu_catalog.element_size() * _gpu_catalog.nelement() / 1024**2:.1f} MB)")
    return _gpu_catalog

def reset_gpu_catalog():
    """Reset the GPU catalog cache (needed when switching catalogs or configs)."""
    global _gpu_catalog
    _gpu_catalog = None

def filter_stars_on_gpu(gpu_catalog, point, radius, max_mag):
    """Filter stars using GPU boolean masking - much faster than CPU Polars"""
    ra, dec = point
    minra, maxra = ra, ra + radius
    mindec, maxdec = dec, dec + radius
    
    mask = (
        (gpu_catalog[:, 0] < maxra) & 
        (gpu_catalog[:, 0] > minra) & 
        (gpu_catalog[:, 1] < maxdec) & 
        (gpu_catalog[:, 1] > mindec) & 
        (gpu_catalog[:, 2] <= max_mag)
    )
    return gpu_catalog[mask]

def vectorized_filter_batch(gpu_catalog, batch_points, config=None, radius=2, max_mag=15):
    """Filter ALL regions in batch simultaneously on GPU - NO Python loop"""
    if config is not None:
        radius = config.search_radius_deg
        max_mag = config.max_mag
    device = gpu_catalog.device
    
    # Convert batch points to tensor
    ra_starts = torch.tensor([p[0] for _, p in batch_points], device=device)
    dec_starts = torch.tensor([p[1] for _, p in batch_points], device=device)
    
    # Broadcast: catalog[N,3] vs batch[B] -> mask[B,N]
    # For each region, check if each star is in bounds
    ra_in_bounds = (gpu_catalog[:, 0].unsqueeze(0) >= ra_starts.unsqueeze(1)) & \
                   (gpu_catalog[:, 0].unsqueeze(0) < (ra_starts + radius).unsqueeze(1))
    dec_in_bounds = (gpu_catalog[:, 1].unsqueeze(0) >= dec_starts.unsqueeze(1)) & \
                    (gpu_catalog[:, 1].unsqueeze(0) < (dec_starts + radius).unsqueeze(1))
    mag_in_bounds = gpu_catalog[:, 2].unsqueeze(0) <= max_mag
    
    # Combined mask [B, N]
    mask = ra_in_bounds & dec_in_bounds & mag_in_bounds
    
    # Return list of star tensors per region
    result = []
    for i in range(len(batch_points)):
        region_stars = gpu_catalog[mask[i]]
        result.append((batch_points[i][0], region_stars))
    
    return result
        
def process(stars, region, point, nr_stars, device='cpu'):
    """Process on specified device - stars can be list or tensor"""
    if not isinstance(stars, torch.Tensor):
        stars = torch.tensor(stars, dtype=torch.float32, device=device)
    else:
        stars = stars.to(device)
    
    scores, points = mass_score_triangle_torch(stars, device=device)
    resultdf = pl.DataFrame({
        "score": scores.cpu().numpy(), 
        "region": [region] * len(scores),
        "stars": points.cpu().numpy()
    })
    final_result_df = resultdf.top_k(5, by="score", reverse=True)
    print(f"Processed {region=} - {point} for {len(stars)} stars, {len(scores):,} triangles")
    return final_result_df

## Result saving

In [9]:
#| export
schema = {
    "score": pl.Float64,
    "region": pl.Float64,
    "stars": pl.List(pl.List(pl.Float64)),
    "tilt": pl.Float64,
}


In [ ]:
#| export
def add_compact_score(df, max_extent_deg, max_mag):
    """Add compact_score column: re-ranks results favoring small, bright, tight patterns."""
    if df.is_empty():
        return df.with_columns(pl.lit(0.0).alias("compact_score"))

    scores = df["score"].to_numpy()
    score_95 = np.percentile(scores, 95) if len(scores) > 1 else scores[0]
    score_95 = max(score_95, 1e-10)
    score_norm = np.clip(scores / score_95, 0, 1)

    def _row_metrics(stars):
        mags = [s[2] for s in stars]
        mean_mag = np.mean(mags)
        std_mag = np.std(mags)
        mag_cv = std_mag / mean_mag if mean_mag > 0 else 0
        brightness_norm = mean_mag / max_mag
        return brightness_norm, mag_cv

    brightness_norms = []
    mag_cvs = []
    for row in df.iter_rows(named=True):
        bn, mcv = _row_metrics(row["stars"])
        brightness_norms.append(bn)
        mag_cvs.append(mcv)

    brightness_norm = np.array(brightness_norms)
    mag_cv = np.array(mag_cvs)

    if "extent_deg" in df.columns:
        extent_norm = np.clip(df["extent_deg"].to_numpy() / max_extent_deg, 0, 1)
    else:
        extents = []
        for row in df.iter_rows(named=True):
            pts = torch.tensor(row["stars"])
            ext = chain_extent_deg(pts) if len(row["stars"]) > 3 else triangle_extent_deg(pts)
            extents.append(ext)
        extent_norm = np.clip(np.array(extents) / max_extent_deg, 0, 1)

    compact = 0.4 * score_norm + 0.25 * extent_norm + 0.25 * brightness_norm + 0.1 * mag_cv
    return df.with_columns(pl.Series("compact_score", compact))


def dedup_results(df):
    """Remove duplicate asterisms by hashing sorted star coordinates."""
    if df.is_empty():
        return df

    hashes = []
    for row in df.iter_rows(named=True):
        coords = sorted((round(s[0], 4), round(s[1], 4)) for s in row["stars"])
        hashes.append(hashlib.sha256(str(coords).encode()).hexdigest())

    df = df.with_columns(pl.Series("_dedup_hash", hashes))
    df = df.sort("score").group_by("_dedup_hash").first().drop("_dedup_hash")
    return df.sort("score")


SHAPE_ABBREV = {
    'triangle': 'tri', 'square': 'sq', 'circle': 'cir',
    'smooth': 'smo', 'smooth_mag': 'smm', 'snake': 'snk',
    'rms': 'rms', 'msd': 'msd', 'bright_line': 'brl',
}

def asterism_id(stars, shape, chain_len=None):
    """Generate a unique hash-based ID for an asterism.

    Format: {shape_abbrev}{chain_len}-{hash8}
    e.g. tri-a3f9b2c1, smo8-7c1d4e90
    """
    coords = sorted((round(s[0], 4), round(s[1], 4)) for s in stars)
    abbrev = SHAPE_ABBREV.get(shape, shape[:3])
    key = f"{abbrev}:{coords}"
    h = hashlib.sha256(key.encode()).hexdigest()[:8]
    if chain_len and chain_len > 3:
        return f"{abbrev}{chain_len}-{h}"
    return f"{abbrev}-{h}"


def assign_asterism_ids(df, shape):
    """Add 'asterism_id' column to a results DataFrame.

    Uses shape name (triangle/square/collinear/circle), not scoring mode,
    so the same stars always get the same ID regardless of 2d/3d scoring.
    For collinear, uses the scoring mode (smooth/snake/etc) since different
    scoring modes find genuinely different chain orderings.
    """
    if df.is_empty():
        return df.with_columns(pl.lit("").alias("asterism_id"))
    ids = []
    for row in df.iter_rows(named=True):
        stars = row['stars']
        chain_len = row.get('chain_len') or len(stars)
        ids.append(asterism_id(stars, shape, chain_len))
    return df.with_columns(pl.Series("asterism_id", ids))


## Region processing pipeline

In [ ]:
#| export
def _score_region(stars_tensor, device, use_hip=False, score_triangles_hip=None,
                  score_squares_hip=None, mode='3d', shape='triangle', max_extent_deg=None,
                  search_radius_deg=None):
    """Score all combinations for a region, return top 5."""
    if shape == 'collinear':
        return score_collinear_region(stars_tensor, device=device, mode=mode)
    if shape == 'square':
        if use_hip and score_squares_hip is not None:
            sq_max_dist = float('inf')
            if max_extent_deg is not None and mode == '2d':
                sq_max_dist = 2.0 * math.sin(math.radians(max_extent_deg) / 2.0)
            scores, points_out = score_squares_hip(stars_tensor, mode=mode, max_dist=sq_max_dist,
                                                          search_radius_deg=search_radius_deg)
        else:
            scores, points_out = mass_score_square_torch(stars_tensor, device=device, mode=mode,
                                                          search_radius_deg=search_radius_deg)
    elif use_hip and score_triangles_hip is not None:
        scores, points_out = score_triangles_hip(stars_tensor, mode=mode,
                                                        search_radius_deg=search_radius_deg)
    else:
        scores, points_out = mass_score_triangle_torch(stars_tensor, device=device, mode=mode,
                                                        search_radius_deg=search_radius_deg)
    k = min(5, len(scores))
    top5_scores, top5_idx = torch.topk(scores, k=k, largest=False)
    top5_points = points_out[top5_idx]
    return top5_scores, top5_points


def _cap_stars(stars_tensor, max_stars):
    """Keep only the brightest max_stars by sorting on magnitude (col 2)."""
    if max_stars <= 0 or len(stars_tensor) <= max_stars:
        return stars_tensor
    _, indices = torch.topk(stars_tensor[:, 2], max_stars, largest=False)
    return stars_tensor[indices]


def _detect_gpu_backend(hip_available=False):
    """Detect GPU and select best backend."""
    if not torch.cuda.is_available():
        return 'cpu', False
    gpu_name = torch.cuda.get_device_name(0).lower()
    is_amd = 'amd' in gpu_name or 'radeon' in gpu_name
    use_hip = hip_available and is_amd
    return 'cuda', use_hip


def process_all_regions(grid, df, config=None, start=0, end=0, batch_size=100, num_streams=4,
                        mode='3d', hip_available=False, score_triangles_hip=None,
                        score_squares_hip=None, shape='triangle'):
    """
    Process regions using the best available GPU backend.
    
    For collinear shape, delegates to process_collinear_regions() which batches
    all angle matrix computation into single GPU passes.
    """
    if end == 0:
        end = len(grid)

    radius = config.search_radius_deg if config else 2
    max_mag = config.max_mag if config else 15
    max_extent = config.max_extent_deg if config else None
    max_stars = config.max_stars_per_region if config else 0
    search_radius = config.search_radius_deg if config else None
    min_stars = 4

    zipped_list = list(zip(range(len(grid)), grid))
    grid_points = zipped_list[start:end]
    global_result = pl.DataFrame()

    device, use_hip = _detect_gpu_backend(hip_available)

    # --- Collinear: use dedicated batched pipeline ---
    if shape == 'collinear':
        if device != 'cuda':
            print("No GPU -- collinear requires CUDA")
            return global_result
        gpu_catalog = load_catalog_to_gpu(df, device)
        gpu_name = torch.cuda.get_device_name(0)
        config_name = config.name if config else "legacy"
        print(f"GPU: {gpu_name}")
        print(f"Config: {config_name} (mag<={max_mag}, radius={radius}deg, extent<={max_extent}deg, max_stars={max_stars})")
        print(f"Shape: collinear (batched GPU pipeline)")
        print(f"Regions: {len(grid_points)}")
        return process_collinear_regions(grid_points, gpu_catalog, config, device, max_extent)

    # --- Circle/Arc: dedicated circumcircle pipeline ---
    if shape == 'circle':
        if device != 'cuda':
            print("No GPU -- circle detection requires CUDA")
            return global_result
        gpu_catalog = load_catalog_to_gpu(df, device)
        gpu_name = torch.cuda.get_device_name(0)
        config_name = config.name if config else "legacy"
        print(f"GPU: {gpu_name}")
        print(f"Config: {config_name} (mag<={max_mag}, radius={radius}deg, extent<={max_extent}deg, max_stars={max_stars})")
        print(f"Shape: circle/arc")
        print(f"Regions: {len(grid_points)}")
        return process_circle_regions(grid_points, gpu_catalog, config, device, max_extent)

    # --- Triangle/Square: existing stream-based pipeline ---
    if device != 'cuda':
        print("No GPU -- falling back to CPU (slow)")
        for idx, point in tqdm.tqdm(grid_points, desc="Regions"):
            stars = stars_for_point_and_radius(df, point=point, radius=radius, max_mag=max_mag)
            if len(stars) > 3:
                result = process(stars.rows(), idx, point, 3, device='cpu')
                if result is not None and not result.is_empty():
                    global_result = result if global_result.is_empty() else global_result.vstack(result)
        return global_result

    gpu_catalog = load_catalog_to_gpu(df, device)
    gpu_name = torch.cuda.get_device_name(0)

    # HIP kernels available for both triangles and squares
    hip_fn_available = (shape == 'triangle' and score_triangles_hip is not None) or \
                       (shape == 'square' and score_squares_hip is not None)
    actual_use_hip = use_hip and hip_fn_available

    if actual_use_hip:
        backend = "HIP kernel"
    else:
        backend = f"CUDA streams ({num_streams}x)"

    config_name = config.name if config else "legacy"
    print(f"GPU: {gpu_name}")
    print(f"Backend: {backend}")
    print(f"Config: {config_name} (mag<={max_mag}, radius={radius}deg, extent<={max_extent}deg, max_stars={max_stars})")
    print(f"Scoring: {mode}, shape: {shape}")
    print(f"Regions: {len(grid_points)}, batch size: {batch_size}")

    streams = [torch.cuda.Stream() for _ in range(num_streams)] if not actual_use_hip else []
    num_batches = (len(grid_points) + batch_size - 1) // batch_size

    for batch_idx in tqdm.tqdm(range(num_batches), desc="Batches"):
        batch_start = batch_idx * batch_size
        batch_end = min(batch_start + batch_size, len(grid_points))
        batch = grid_points[batch_start:batch_end]

        try:
            filtered_regions = vectorized_filter_batch(gpu_catalog, batch, config=config,
                                                       radius=radius, max_mag=max_mag)

            region_results = []

            if actual_use_hip:
                for idx, stars_tensor in filtered_regions:
                    stars_tensor = _cap_stars(stars_tensor, max_stars)
                    if len(stars_tensor) >= min_stars:
                        top5_s, top5_p = _score_region(stars_tensor, device, use_hip=True,
                                                       score_triangles_hip=score_triangles_hip,
                                                       score_squares_hip=score_squares_hip,
                                                       mode=mode, shape=shape,
                                                       max_extent_deg=max_extent,
                                                       search_radius_deg=search_radius)
                        region_results.append((idx, top5_s, top5_p))
                        del stars_tensor
            else:
                regions_per_stream = (len(filtered_regions) + num_streams - 1) // num_streams
                stream_results = [[] for _ in range(num_streams)]

                for stream_idx, stream in enumerate(streams):
                    s_start = stream_idx * regions_per_stream
                    s_end = min(s_start + regions_per_stream, len(filtered_regions))
                    if s_start >= len(filtered_regions):
                        break
                    with torch.cuda.stream(stream):
                        for i in range(s_start, s_end):
                            idx, stars_tensor = filtered_regions[i]
                            stars_tensor = _cap_stars(stars_tensor, max_stars)
                            if len(stars_tensor) >= min_stars:
                                top5_s, top5_p = _score_region(stars_tensor, device, use_hip=False,
                                                               mode=mode, shape=shape,
                                                               max_extent_deg=max_extent,
                                                               search_radius_deg=search_radius)
                                stream_results[stream_idx].append((idx, top5_s, top5_p))
                                del stars_tensor

                for stream in streams:
                    stream.synchronize()

                for sr in stream_results:
                    region_results.extend(sr)

            if len(region_results) == 0:
                torch.cuda.empty_cache()
                continue

            all_scores = []
            all_points = []
            all_regions = []
            for idx, scores, points in region_results:
                n = len(scores)
                all_scores.append(scores)
                all_points.append(points)
                all_regions.append(torch.full((n,), idx, dtype=torch.int32, device=device))

            scores_cat = torch.cat(all_scores, dim=0)
            points_cat = torch.cat(all_points, dim=0)
            regions_cat = torch.cat(all_regions, dim=0)
            del all_scores, all_points, all_regions, region_results

            if max_extent is not None and len(points_cat) > 0:
                extents = shape_extent_deg_batch(points_cat)
                extent_mask = extents <= max_extent
                scores_cat = scores_cat[extent_mask]
                points_cat = points_cat[extent_mask]
                regions_cat = regions_cat[extent_mask]

            if len(scores_cat) == 0:
                torch.cuda.empty_cache()
                continue

            tilt_cat = compute_tilt_batch(points_cat, search_radius_deg=search_radius)

            batch_df = pl.DataFrame({
                "score": scores_cat.cpu().numpy(),
                "region": regions_cat.cpu().numpy(),
                "stars": points_cat.cpu().numpy(),
                "tilt": tilt_cat.cpu().numpy(),
            })

            del scores_cat, points_cat, regions_cat
            torch.cuda.empty_cache()

            global_result = batch_df if global_result.is_empty() else pl.concat([global_result, batch_df])
            del batch_df

        except Exception as e:
            print(f"Batch {batch_idx + 1} error: {e}")
            torch.cuda.empty_cache()
            continue

    return global_result

In [11]:
#| hide
import nbdev; nbdev.nbdev_export()